<a href="https://colab.research.google.com/github/hahmedhh/EEG/blob/main/eeg_models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [ ]:
pip install mne -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 100.9 MB/s eta 0:00:00


In [ ]:
pip install numpy pandas scipy scikit-learn xgboost mne matplotlib -q

In [ ]:
pip install tensorflow scikit-learn pandas matplotlib seaborn scipy -q

In [ ]:
pip install tensorflow mne -q

In [ ]:
pip install catboost -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 9.7 MB/s eta 0:00:00


In [ ]:
# ==========================================
# CELL 1: OPTIMIZED AttentionEEGNet (MSA + MixUp)
# ==========================================
import os
import numpy as np
import scipy.io
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from scipy.signal import butter, lfilter
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import cohen_kappa_score, f1_score
import time
import json
import copy
import math

# ==================== CONFIG ====================
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Device: {DEVICE}")

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

FS = 250
N_CLASSES = 4
N_CHANNELS = 22
TIME_WINDOW = 1000

# SOTA Training Params
USE_TTA = True
LABEL_SMOOTHING = 0.2     # Increased smoothing
BATCH_SIZE = 64
LR_PRETRAIN = 0.001       # Start higher for AdamW
LR_FINETUNE = 0.0001      # Lower for fine-tuning
WEIGHT_DECAY = 0.01       # Critical for Attention models
EPOCHS_PRETRAIN = 100
EPOCHS_FINETUNE = 60
MIXUP_ALPHA = 0.5         # MixUp strength

DATA_PATH = '/content/gdrive/MyDrive/BCICIV-2a-mat'
MODEL_NAME = 'AttentionEEGNet_Optimized'
SAVE_PATH = f'/content/gdrive/MyDrive/BCI_ENSEMBLE/{MODEL_NAME}'

if not os.path.exists(SAVE_PATH):
    os.makedirs(SAVE_PATH)

# ==================== UTILS ====================
def butter_bandpass_filter(data, lowcut=4.0, highcut=38.0, fs=250, order=5):
    nyq = 0.5 * fs
    b, a = butter(order, [lowcut/nyq, highcut/nyq], btype='band')
    return lfilter(b, a, data, axis=-1)

def euclidean_alignment(X_data):
    print(f"   ... Applying Euclidean Alignment (Shape: {X_data.shape})")
    covariances = np.matmul(X_data, np.transpose(X_data, (0, 2, 1)))
    mean_cov = np.mean(covariances, axis=0)
    d, v = np.linalg.eigh(mean_cov)
    d_inv_sqrt = np.diag(1.0 / np.sqrt(d + 1e-7))
    whitening_mat = np.dot(v, np.dot(d_inv_sqrt, v.T))
    X_transposed = np.transpose(X_data, (0, 2, 1))
    X_aligned = np.matmul(X_transposed, whitening_mat)
    return np.transpose(X_aligned, (0, 2, 1))

def scale_data(X, scaler, fit=False):
    n, c, t = X.shape
    x_flat = X.transpose(0, 2, 1).reshape(-1, c)
    if fit:
        x_scaled = scaler.fit_transform(x_flat)
    else:
        x_scaled = scaler.transform(x_flat)
    return x_scaled.reshape(n, t, c).transpose(0, 2, 1)

def load_bci_data_raw(subject_id, base_path, is_train=True):
    file_type = 'T' if is_train else 'E'
    file_name = f"A{subject_id:02d}{file_type}.mat"
    full_path = os.path.join(base_path, file_name)

    if not os.path.exists(full_path):
        file_name = f"A0{subject_id}{file_type}.mat"
        full_path = os.path.join(base_path, file_name)
        if not os.path.exists(full_path): return None, None

    try:
        mat = scipy.io.loadmat(full_path)
        data_struct = mat['data']
        all_X, all_y = [], []

        for i in range(data_struct.shape[1]):
            try:
                run_data = data_struct[0][i]
                X_cnt = run_data['X'][0][0]
                trial_idx = run_data['trial'][0][0].flatten()
                y_cnt = run_data['y'][0][0].flatten()

                if len(trial_idx) == 0: continue

                for j, start_idx in enumerate(trial_idx):
                    label = y_cnt[j]
                    if label not in [1, 2, 3, 4]: continue
                    # Sliding window TTA
                    offsets = [0, int(0.25*FS), int(0.5*FS)] if is_train else [0]
                    for off in offsets:
                        s = (start_idx - 1) + off
                        e = s + TIME_WINDOW
                        if e <= X_cnt.shape[0]:
                            raw_epoch = X_cnt[s:e, :22].T
                            filtered = butter_bandpass_filter(raw_epoch, 4.0, 38.0, FS, 4)
                            all_X.append(filtered)
                            all_y.append(label - 1)
            except: continue

        if len(all_X) == 0: return None, None
        return np.stack(all_X), np.array(all_y)
    except: return None, None

def get_dataloader(X, y, batch_size, shuffle=True):
    tensor_x = torch.Tensor(X).unsqueeze(1)
    tensor_y = torch.LongTensor(y)
    return DataLoader(TensorDataset(tensor_x, tensor_y), batch_size=batch_size, shuffle=shuffle)

# ==================== MIXUP AUGMENTATION ====================
def mixup_data(x, y, alpha=0.5):
    '''Returns mixed inputs, pairs of targets, and lambda'''
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1
    batch_size = x.size(0)
    index = torch.randperm(batch_size).to(x.device)
    mixed_x = lam * x + (1 - lam) * x[index, :]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

def train_epoch(model, loader, optimizer, criterion, scheduler=None):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for inputs, labels in loader:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)

        # Apply MixUp
        inputs, targets_a, targets_b, lam = mixup_data(inputs, labels, MIXUP_ALPHA)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = mixup_criterion(criterion, outputs, targets_a, targets_b, lam)
        loss.backward()

        # Gradient Clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        optimizer.step()
        if scheduler: scheduler.step()

        total_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        # Approximate accuracy for MixUp
        correct += (lam * predicted.eq(targets_a.data).cpu().sum().float()
                    + (1 - lam) * predicted.eq(targets_b.data).cpu().sum().float())

    return total_loss / len(loader), 100 * correct / total

def evaluate_probs(model, loader):
    model.eval()
    all_probs, all_preds, all_labels = [], [], []
    with torch.no_grad():
        for inputs, labels in loader:
            inputs = inputs.to(DEVICE)
            outputs = model(inputs)
            probs = torch.softmax(outputs, dim=1)
            _, preds = torch.max(outputs, 1)
            all_probs.append(probs.cpu().numpy())
            all_preds.append(preds.cpu().numpy())
            all_labels.append(labels.numpy())
    return (np.vstack(all_probs), np.concatenate(all_preds), np.concatenate(all_labels))

# ==================== OPTIMIZED MODEL: MSA-EEGNet ====================
#
class MSA_EEGNet(nn.Module):
    """
    Replaces custom attention with standard Multi-Head Self-Attention (MSA)
    and uses a Depthwise-Separable Conv backbone.
    """
    def __init__(self, n_classes=4, n_channels=22, n_time=1000):
        super(MSA_EEGNet, self).__init__()

        # --- 1. Convolutional Backbone (Feature Extraction) ---
        self.F1 = 8
        self.D = 2
        self.F2 = 16

        # Temporal Conv
        self.conv1 = nn.Conv2d(1, self.F1, (1, 64), padding=(0, 32), bias=False)
        self.bn1 = nn.BatchNorm2d(self.F1)

        # Spatial Conv (Depthwise)
        self.conv2 = nn.Conv2d(self.F1, self.F1 * self.D, (n_channels, 1), groups=self.F1, bias=False)
        self.bn2 = nn.BatchNorm2d(self.F1 * self.D)
        self.act1 = nn.ELU()
        self.pool1 = nn.AvgPool2d((1, 4))
        self.drop1 = nn.Dropout(0.5)

        # Separable Conv
        self.conv3 = nn.Conv2d(self.F1*self.D, self.F1*self.D, (1, 16), padding=(0, 8), groups=self.F1*self.D, bias=False)
        self.conv4 = nn.Conv2d(self.F1*self.D, self.F2, (1, 1), bias=False)
        self.bn3 = nn.BatchNorm2d(self.F2)
        self.act2 = nn.ELU()
        self.pool2 = nn.AvgPool2d((1, 8))
        self.drop2 = nn.Dropout(0.5)

        # --- 2. Multi-Head Self Attention (Global Context) ---
        self.embed_dim = self.F2 # 16 features
        self.num_heads = 4
        self.mha = nn.MultiheadAttention(embed_dim=self.embed_dim, num_heads=self.num_heads, dropout=0.5)
        self.norm_attn = nn.LayerNorm(self.embed_dim)

        # Output dimension calculation
        # 1000 -> pool4 -> 250 -> pool8 -> ~31 time steps
        self.flatten_dim = self.F2 * 31

        # Classifier
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(self.flatten_dim, n_classes)
        )

    def forward(self, x):
        # x: (B, 1, 22, 1000)

        # Backbone
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.conv2(x)
        x = self.bn2(x)
        x = self.act1(x)
        x = self.pool1(x)
        x = self.drop1(x)

        x = self.conv3(x)
        x = self.conv4(x)
        x = self.bn3(x)
        x = self.act2(x)
        x = self.pool2(x)
        x = self.drop2(x) # (B, 16, 1, 31)

        # Prepare for Attention: (Sequence, Batch, Features)
        x = x.squeeze(2) # (B, 16, 31)
        x = x.permute(2, 0, 1) # (31, B, 16)

        # Attention Block
        attn_out, _ = self.mha(x, x, x)
        x = x + attn_out # Residual
        x = self.norm_attn(x)

        # Prepare for classifier: (B, 16, 31)
        x = x.permute(1, 2, 0)
        x = self.classifier(x)
        return x

# ==================== MAIN LOOP ====================
print(f"\n{'='*60}")
print(f"STARTING {MODEL_NAME} - MSA + MIXUP")
print(f"{'='*60}")

# Cache Data
data_cache = {}
print("📥 Caching Raw Data...")
all_subjects = list(range(1, 10))
for s in all_subjects:
    X_tr, y_tr = load_bci_data_raw(s, DATA_PATH, True)
    X_te, y_te = load_bci_data_raw(s, DATA_PATH, False)
    if X_tr is not None:
        data_cache[s] = {'X_tr': X_tr, 'y_tr': y_tr, 'X_te': X_te, 'y_te': y_te}

results = {}
kappa_scores = {}

for target_sub in all_subjects:
    start_time = time.time()
    print(f"\n🎯 TARGET SUBJECT: {target_sub}")

    # Prepare Data
    X_source_list, y_source_list = [], []
    for src in all_subjects:
        if src != target_sub and src in data_cache:
            X_source_list.append(data_cache[src]['X_tr'])
            y_source_list.append(data_cache[src]['y_tr'])

    if not X_source_list: continue
    X_source = np.concatenate(X_source_list)
    y_source = np.concatenate(y_source_list)

    X_tgt_tr = data_cache[target_sub]['X_tr']
    y_tgt_tr = data_cache[target_sub]['y_tr']
    X_tgt_te = data_cache[target_sub]['X_te']
    y_tgt_te = data_cache[target_sub]['y_te']

    print("   ⚙️ Euclidean Alignment & Scaling...")
    X_source = euclidean_alignment(X_source)
    X_tgt_tr = euclidean_alignment(X_tgt_tr)
    X_tgt_te = euclidean_alignment(X_tgt_te)

    scaler = StandardScaler()
    X_source = scale_data(X_source, scaler, fit=True)
    X_tgt_tr = scale_data(X_tgt_tr, scaler, fit=False)
    X_tgt_te = scale_data(X_tgt_te, scaler, fit=False)

    # Dataloaders
    train_loader = get_dataloader(X_source, y_source, BATCH_SIZE)
    ft_loader = get_dataloader(X_tgt_tr, y_tgt_tr, 32) # Smaller batch for FT stability
    test_loader = get_dataloader(X_tgt_te, y_tgt_te, 32, shuffle=False)

    # --- 1. PRE-TRAINING ---
    print(f"   🚀 Pre-training...")
    model = MSA_EEGNet(n_classes=N_CLASSES).to(DEVICE)
    criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)

    # Use AdamW + OneCycle for better convergence
    optimizer_pre = optim.AdamW(model.parameters(), lr=LR_PRETRAIN, weight_decay=WEIGHT_DECAY)
    scheduler_pre = optim.lr_scheduler.OneCycleLR(optimizer_pre, max_lr=LR_PRETRAIN,
                                                  steps_per_epoch=len(train_loader), epochs=EPOCHS_PRETRAIN)

    for ep in range(EPOCHS_PRETRAIN):
        loss, acc = train_epoch(model, train_loader, optimizer_pre, criterion, scheduler_pre)
        if (ep+1) % 50 == 0:
            print(f"     Ep {ep+1}: Loss {loss:.4f} | Acc {acc:.2f}%")

    # --- 2. FINE-TUNING (SAVE BEST) ---
    print(f"   🔧 Fine-tuning (MixUp + Save Best)...")
    optimizer_ft = optim.AdamW(model.parameters(), lr=LR_FINETUNE, weight_decay=WEIGHT_DECAY)

    best_ft_acc = 0.0
    best_model_state = copy.deepcopy(model.state_dict())

    for ep in range(EPOCHS_FINETUNE):
        loss, acc = train_epoch(model, ft_loader, optimizer_ft, criterion)

        # Save best model state
        if acc > best_ft_acc:
            best_ft_acc = acc
            best_model_state = copy.deepcopy(model.state_dict())

        if (ep+1) % 10 == 0:
             print(f"     Ep {ep+1}: Loss {loss:.4f} | Acc {acc:.2f}% | Best {best_ft_acc:.2f}%")

    model.load_state_dict(best_model_state)

    # --- 3. EVALUATION ---
    try:
        probs, preds, labels = evaluate_probs(model, test_loader)
        acc = 100 * (preds == labels).mean()
        kappa = cohen_kappa_score(labels, preds)
        f1 = f1_score(labels, preds, average='weighted')

        # Save results
        save_file = os.path.join(SAVE_PATH, f"S{target_sub}")
        np.save(f"{save_file}_probs.npy", probs)
        np.save(f"{save_file}_labels.npy", labels)
        np.save(f"{save_file}_preds.npy", preds)
        np.save(f"{save_file}_metrics.npy", np.array([acc, kappa, f1]))

        results[target_sub] = acc
        kappa_scores[target_sub] = kappa
        print(f"   📊 Result: Acc {acc:.2f}% | Kappa {kappa:.3f}")
    except Exception as e:
        print(f"   ❌ Eval Failed: {e}")

# ==================== SUMMARY ====================
print(f"\n{'='*60}")
print(f"FINAL RESULTS: {MODEL_NAME}")
if results:
    accs = list(results.values())
    for sub in all_subjects:
        if sub in results:
            print(f"S{sub}: {results[sub]:.2f}% (K={kappa_scores[sub]:.3f})")
    print("-" * 35)
    print(f"AVG: {np.mean(accs):.2f}% | STD: {np.std(accs):.2f}")

    with open(os.path.join(SAVE_PATH, 'summary.json'), 'w') as f:
        json.dump({'acc': results, 'kappa': kappa_scores, 'mean': np.mean(accs)}, f)
    print(f"✅ Training completed! Results saved to {SAVE_PATH}/")

✅ Device: cuda

STARTING AttentionEEGNet_Optimized - MSA + MIXUP
📥 Caching Raw Data...

🎯 TARGET SUBJECT: 1
   ⚙️ Euclidean Alignment & Scaling...
   ... Applying Euclidean Alignment (Shape: (6912, 22, 1000))
   ... Applying Euclidean Alignment (Shape: (864, 22, 1000))
   ... Applying Euclidean Alignment (Shape: (288, 22, 1000))
   🚀 Pre-training...
     Ep 50: Loss 1.2260 | Acc 51.77%
     Ep 100: Loss 1.1934 | Acc 55.12%
   🔧 Fine-tuning (MixUp + Save Best)...
     Ep 10: Loss 1.1120 | Acc 62.76% | Best 62.76%
     Ep 20: Loss 1.1056 | Acc 62.45% | Best 64.66%
     Ep 30: Loss 1.0691 | Acc 67.52% | Best 68.74%
     Ep 40: Loss 1.0729 | Acc 67.05% | Best 70.00%
     Ep 50: Loss 1.0395 | Acc 68.57% | Best 71.42%
     Ep 60: Loss 1.0546 | Acc 68.79% | Best 76.35%
   📊 Result: Acc 76.74% | Kappa 0.690

🎯 TARGET SUBJECT: 2
   ⚙️ Euclidean Alignment & Scaling...
   ... Applying Euclidean Alignment (Shape: (6912, 22, 1000))
   ... Applying Euclidean Alignment (Shape: (864, 22, 1000))
   ...

In [ ]:
# ==========================================
# CELL 2: ATTENTION CRNN (SOTA RNN)
# ==========================================
import os
import numpy as np
import scipy.io
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from scipy.signal import butter, lfilter
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import cohen_kappa_score, f1_score
import time
import json
import copy

# ==================== CONFIGURATION ====================
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Device: {DEVICE}")

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Data Params
FS = 250
N_CLASSES = 4
N_CHANNELS = 22
TIME_WINDOW = 1000

# Training Params (Optimized for Attention Models)
BATCH_SIZE = 64
LR_PRETRAIN = 0.001
LR_FINETUNE = 0.0001
EPOCHS_PRETRAIN = 100
EPOCHS_FINETUNE = 50
DROPOUT_RATE = 0.5
RNN_HIDDEN = 64

DATA_PATH = '/content/gdrive/MyDrive/BCICIV-2a-mat'
MODEL_NAME = 'Attention_CRNN'
SAVE_PATH = f'/content/gdrive/MyDrive/BCI_ENSEMBLE/{MODEL_NAME}'

if not os.path.exists(SAVE_PATH):
    os.makedirs(SAVE_PATH)

# ==================== UTILS ====================
def butter_bandpass_filter(data, lowcut=4.0, highcut=38.0, fs=250, order=5):
    nyq = 0.5 * fs
    b, a = butter(order, [lowcut/nyq, highcut/nyq], btype='band')
    return lfilter(b, a, data, axis=-1)

def euclidean_alignment(X_data):
    print(f"   ... Applying Euclidean Alignment (Shape: {X_data.shape})")
    covariances = np.matmul(X_data, np.transpose(X_data, (0, 2, 1)))
    mean_cov = np.mean(covariances, axis=0)
    d, v = np.linalg.eigh(mean_cov)
    d_inv_sqrt = np.diag(1.0 / np.sqrt(d + 1e-7))
    whitening_mat = np.dot(v, np.dot(d_inv_sqrt, v.T))
    X_transposed = np.transpose(X_data, (0, 2, 1))
    X_aligned = np.matmul(X_transposed, whitening_mat)
    return np.transpose(X_aligned, (0, 2, 1))

def scale_data(X, scaler, fit=False):
    n, c, t = X.shape
    x_flat = X.transpose(0, 2, 1).reshape(-1, c)
    if fit:
        x_scaled = scaler.fit_transform(x_flat)
    else:
        x_scaled = scaler.transform(x_flat)
    return x_scaled.reshape(n, t, c).transpose(0, 2, 1)

def load_bci_data_raw(subject_id, base_path, is_train=True):
    file_type = 'T' if is_train else 'E'
    file_name = f"A{subject_id:02d}{file_type}.mat"
    full_path = os.path.join(base_path, file_name)

    if not os.path.exists(full_path):
        file_name = f"A0{subject_id}{file_type}.mat"
        full_path = os.path.join(base_path, file_name)
        if not os.path.exists(full_path): return None, None

    try:
        mat = scipy.io.loadmat(full_path)
        data_struct = mat['data']
        all_X, all_y = [], []
        for i in range(data_struct.shape[1]):
            try:
                run_data = data_struct[0][i]
                X_cnt = run_data['X'][0][0]
                trial_idx = run_data['trial'][0][0].flatten()
                y_cnt = run_data['y'][0][0].flatten()
                if len(trial_idx) == 0: continue
                for j, start_idx in enumerate(trial_idx):
                    label = y_cnt[j]
                    if label not in [1, 2, 3, 4]: continue
                    # TTA: Simple sliding window
                    offsets = [0, int(0.25*FS), int(0.5*FS)] if is_train else [0]
                    for off in offsets:
                        s = (start_idx - 1) + off
                        e = s + TIME_WINDOW
                        if e <= X_cnt.shape[0]:
                            raw_epoch = X_cnt[s:e, :22].T
                            filtered = butter_bandpass_filter(raw_epoch, 4.0, 38.0, FS, 4)
                            all_X.append(filtered)
                            all_y.append(label - 1)
            except: continue
        if len(all_X) == 0: return None, None
        return np.stack(all_X), np.array(all_y)
    except: return None, None

def get_dataloader(X, y, batch_size, shuffle=True):
    tensor_x = torch.Tensor(X).unsqueeze(1) # (B, 1, 22, 1000)
    tensor_y = torch.LongTensor(y)
    return DataLoader(TensorDataset(tensor_x, tensor_y), batch_size=batch_size, shuffle=shuffle)

def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for inputs, labels in loader:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    return total_loss / len(loader), 100 * correct / total

def evaluate_probs(model, loader):
    model.eval()
    all_probs, all_preds, all_labels = [], [], []
    with torch.no_grad():
        for inputs, labels in loader:
            inputs = inputs.to(DEVICE)
            outputs = model(inputs)
            probs = torch.softmax(outputs, dim=1)
            _, preds = torch.max(outputs, 1)
            all_probs.append(probs.cpu().numpy())
            all_preds.append(preds.cpu().numpy())
            all_labels.append(labels.numpy())
    return (np.vstack(all_probs), np.concatenate(all_preds), np.concatenate(all_labels))

# ==================== ATTENTION-CRNN ARCHITECTURE ====================
class AttentionBlock(nn.Module):
    """
    Computes a weighted average of the RNN output (Attention Context)
    """
    def __init__(self, hidden_dim):
        super(AttentionBlock, self).__init__()
        self.hidden_dim = hidden_dim
        self.projection = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, rnn_outputs):
        # rnn_outputs: (Batch, Seq, Hidden)

        # Calculate scores (Batch, Seq, 1)
        energy = self.projection(rnn_outputs)
        weights = F.softmax(energy, dim=1)

        # Weighted sum (Batch, Hidden)
        # Bmm: (Batch, 1, Seq) x (Batch, Seq, Hidden) -> (Batch, 1, Hidden)
        context = torch.bmm(weights.permute(0, 2, 1), rnn_outputs).squeeze(1)

        return context, weights

class AttentionCRNN(nn.Module):
    def __init__(self, n_classes=4, n_channels=22, n_time=1000):
        super(AttentionCRNN, self).__init__()

        # 1. Convolutional Feature Extractor
        # Downsamples time (1000 -> ~62) and extracts spatial features
        self.conv_stem = nn.Sequential(
            nn.Conv2d(1, 32, (1, 25), padding=(0, 12)),
            nn.BatchNorm2d(32),
            nn.ELU(),
            nn.AvgPool2d((1, 4)),

            nn.Conv2d(32, 64, (n_channels, 1)),
            nn.BatchNorm2d(64),
            nn.ELU(),
            nn.AvgPool2d((1, 4)),
            nn.Dropout(DROPOUT_RATE)
        )

        # 2. Bi-LSTM Layer
        # Input: 64 features per time step
        self.rnn_input_size = 64
        self.hidden_dim = RNN_HIDDEN

        self.lstm = nn.LSTM(
            input_size=self.rnn_input_size,
            hidden_size=self.hidden_dim,
            num_layers=2,
            batch_first=True,
            bidirectional=True,
            dropout=0.5
        )

        # 3. Attention Mechanism
        # Bidirectional doubles the hidden size
        self.attention = AttentionBlock(self.hidden_dim * 2)

        # 4. Classifier
        self.fc = nn.Sequential(
            nn.Linear(self.hidden_dim * 2, 64),
            nn.ELU(),
            nn.Dropout(0.5),
            nn.Linear(64, n_classes)
        )

    def forward(self, x):
        # Input: (B, 1, 22, 1000)

        # --- CNN Stem ---
        x = self.conv_stem(x) # (B, 64, 1, 62)
        x = x.squeeze(2)      # (B, 64, 62)
        x = x.permute(0, 2, 1) # (B, 62, 64) -> (Batch, Seq, Features)

        # --- Bi-LSTM ---
        # rnn_out: (Batch, Seq, Hidden*2)
        rnn_out, _ = self.lstm(x)

        # --- Attention ---
        # context: (Batch, Hidden*2)
        context, weights = self.attention(rnn_out)

        # --- Classification ---
        out = self.fc(context)
        return out

# ==================== MAIN LOOP ====================
print(f"\n{'='*60}")
print(f"STARTING {MODEL_NAME} - ATTENTION IS ALL YOU NEED")
print(f"{'='*60}")

# Cache Data
data_cache = {}
print("📥 Caching Raw Data...")
all_subjects = list(range(1, 10))
for s in all_subjects:
    X_tr, y_tr = load_bci_data_raw(s, DATA_PATH, True)
    X_te, y_te = load_bci_data_raw(s, DATA_PATH, False)
    if X_tr is not None:
        data_cache[s] = {'X_tr': X_tr, 'y_tr': y_tr, 'X_te': X_te, 'y_te': y_te}

results = {}
kappa_scores = {}

for target_sub in all_subjects:
    start_time = time.time()
    print(f"\n🎯 TARGET SUBJECT: {target_sub}")

    # Prepare Data
    X_source_list, y_source_list = [], []
    for src in all_subjects:
        if src != target_sub and src in data_cache:
            X_source_list.append(data_cache[src]['X_tr'])
            y_source_list.append(data_cache[src]['y_tr'])

    if not X_source_list: continue
    X_source = np.concatenate(X_source_list)
    y_source = np.concatenate(y_source_list)

    X_tgt_tr = data_cache[target_sub]['X_tr']
    y_tgt_tr = data_cache[target_sub]['y_tr']
    X_tgt_te = data_cache[target_sub]['X_te']
    y_tgt_te = data_cache[target_sub]['y_te']

    print("   ⚙️ Euclidean Alignment...")
    X_source = euclidean_alignment(X_source)
    X_tgt_tr = euclidean_alignment(X_tgt_tr)
    X_tgt_te = euclidean_alignment(X_tgt_te)

    scaler = StandardScaler()
    X_source = scale_data(X_source, scaler, fit=True)
    X_tgt_tr = scale_data(X_tgt_tr, scaler, fit=False)
    X_tgt_te = scale_data(X_tgt_te, scaler, fit=False)

    train_loader = get_dataloader(X_source, y_source, BATCH_SIZE)
    ft_loader = get_dataloader(X_tgt_tr, y_tgt_tr, 32)
    test_loader = get_dataloader(X_tgt_te, y_tgt_te, 32, shuffle=False)

    # --- TRAIN ---
    print(f"   🚀 Pre-training...")
    model = AttentionCRNN(n_classes=N_CLASSES).to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=LR_PRETRAIN, weight_decay=0.01)

    for ep in range(EPOCHS_PRETRAIN):
        loss, acc = train_epoch(model, train_loader, optimizer, criterion)
        if (ep+1) % 50 == 0:
            print(f"     Ep {ep+1}: Loss {loss:.4f} | Acc {acc:.2f}%")

    print(f"   🔧 Fine-tuning (Saving Best)...")
    optimizer_ft = optim.AdamW(model.parameters(), lr=LR_FINETUNE, weight_decay=0.01)

    best_acc = 0.0
    best_state = copy.deepcopy(model.state_dict())

    for ep in range(EPOCHS_FINETUNE):
        loss, acc = train_epoch(model, ft_loader, optimizer_ft, criterion)
        if acc > best_acc:
            best_acc = acc
            best_state = copy.deepcopy(model.state_dict())
        if (ep+1) % 10 == 0:
            print(f"     FT Ep {ep+1}: Loss {loss:.4f} | Acc {acc:.2f}%")

    model.load_state_dict(best_state)

    # --- EVAL ---
    probs, preds, labels = evaluate_probs(model, test_loader)
    acc = 100 * (preds == labels).mean()
    kappa = cohen_kappa_score(labels, preds)

    results[target_sub] = acc
    kappa_scores[target_sub] = kappa
    print(f"   📊 S{target_sub}: Acc {acc:.2f}% | Kappa {kappa:.3f}")

    # Save
    s_path = os.path.join(SAVE_PATH, f"S{target_sub}")
    np.save(f"{s_path}_probs.npy", probs)
    np.save(f"{s_path}_labels.npy", labels)
    np.save(f"{s_path}_preds.npy", preds)
    np.save(f"{s_path}_metrics.npy", np.array([acc, kappa]))

# ==================== SUMMARY ====================
print(f"\n{'='*60}")
print(f"FINAL RESULTS: {MODEL_NAME}")
if results:
    accs = list(results.values())
    print(f"AVG: {np.mean(accs):.2f}% | STD: {np.std(accs):.2f}")
    with open(os.path.join(SAVE_PATH, 'summary.json'), 'w') as f:
        json.dump({'acc': results, 'kappa': kappa_scores, 'mean': np.mean(accs)}, f)
    print(f"✅ Saved to {SAVE_PATH}/")

✅ Device: cuda

STARTING Attention_CRNN - ATTENTION IS ALL YOU NEED
📥 Caching Raw Data...

🎯 TARGET SUBJECT: 1
   ⚙️ Euclidean Alignment...
   ... Applying Euclidean Alignment (Shape: (6912, 22, 1000))
   ... Applying Euclidean Alignment (Shape: (864, 22, 1000))
   ... Applying Euclidean Alignment (Shape: (288, 22, 1000))
   🚀 Pre-training...
     Ep 50: Loss 0.2258 | Acc 91.65%
     Ep 100: Loss 0.0848 | Acc 97.06%
   🔧 Fine-tuning (Saving Best)...
     FT Ep 10: Loss 0.6289 | Acc 76.39%
     FT Ep 20: Loss 0.3816 | Acc 85.42%
     FT Ep 30: Loss 0.1935 | Acc 92.13%
     FT Ep 40: Loss 0.1488 | Acc 94.68%
     FT Ep 50: Loss 0.0929 | Acc 97.11%
   📊 S1: Acc 72.92% | Kappa 0.639

🎯 TARGET SUBJECT: 2
   ⚙️ Euclidean Alignment...
   ... Applying Euclidean Alignment (Shape: (6912, 22, 1000))
   ... Applying Euclidean Alignment (Shape: (864, 22, 1000))
   ... Applying Euclidean Alignment (Shape: (288, 22, 1000))
   🚀 Pre-training...
     Ep 50: Loss 0.2064 | Acc 92.55%
     Ep 100: Loss 0.

In [ ]:
# ==========================================
# CELL 3: EEGInception TRANSFORMER MODEL
# ==========================================
import os
import numpy as np
import scipy.io as sio
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input, Conv2D, DepthwiseConv2D, Concatenate,
                                     BatchNormalization, Activation, AveragePooling2D,
                                     Flatten, Dense, Dropout)
from tensorflow.keras.constraints import MaxNorm
from scipy.signal import butter, lfilter
import json
from sklearn.metrics import cohen_kappa_score, f1_score
import time

# ==================== CONFIG ====================
MODEL_NAME = 'EEGInception'
SAVE_PATH = f'/content/gdrive/MyDrive/BCI_ENSEMBLE/{MODEL_NAME}'
DATA_PATH = "/content/gdrive/MyDrive/BCICIV-2a-mat"
FS = 250
N_CLASSES = 4
N_CHANNELS = 22
TIME_WINDOW = 1000
BATCH_SIZE = 64
USE_TTA = True  # Test Time Augmentation

# Create save directory
os.makedirs(SAVE_PATH, exist_ok=True)

# ==================== UTILITY FUNCTIONS ====================
def butter_bandpass_filter(data, lowcut=4.0, highcut=38.0, fs=250, order=5):
    nyq = 0.5 * fs
    b, a = butter(order, [lowcut/nyq, highcut/nyq], btype='band')
    return lfilter(b, a, data, axis=-1)

def euclidean_alignment(X_data):
    """
    Apply Euclidean Alignment to EEG data
    """
    print(f"   ... Applying Euclidean Alignment (Shape: {X_data.shape})")
    # Calculate covariance matrices
    covariances = np.matmul(X_data, np.transpose(X_data, (0, 2, 1)))
    mean_cov = np.mean(covariances, axis=0)

    # Eigen decomposition
    d, v = np.linalg.eigh(mean_cov)
    d_inv_sqrt = np.diag(1.0 / np.sqrt(d + 1e-7))
    whitening_mat = np.dot(v, np.dot(d_inv_sqrt, v.T))

    # Apply whitening
    X_transposed = np.transpose(X_data, (0, 2, 1))
    X_aligned = np.matmul(X_transposed, whitening_mat)
    return np.transpose(X_aligned, (0, 2, 1))

def scale_data(X, scaler=None, fit=False):
    """
    Apply standardization to EEG data
    """
    n, c, t = X.shape
    x_flat = X.transpose(0, 2, 1).reshape(-1, c)

    if fit:
        from sklearn.preprocessing import StandardScaler
        scaler = StandardScaler()
        x_scaled = scaler.fit_transform(x_flat)
    else:
        x_scaled = scaler.transform(x_flat)

    return x_scaled.reshape(n, t, c).transpose(0, 2, 1), scaler

def _load_raw_data_from_mat_file(file_path, fs, time_window, is_train=True):
    """
    Helper function to load and preprocess data from a single .mat file
    with a nested 'data' structure.
    """
    all_X_epochs, all_y_labels = [], []
    try:
        mat = sio.loadmat(file_path)
        data_struct = mat['data']  # Expecting this nested structure

        for i in range(data_struct.shape[1]):  # Iterate through runs/epochs
            run_data = data_struct[0][i]
            X_cnt = run_data['X'][0][0]  # Raw EEG data (samples x channels)
            trial_idx = run_data['trial'][0][0].flatten()  # Start indices of trials
            y_cnt = run_data['y'][0][0].flatten()  # Labels for trials

            for j, start_idx in enumerate(trial_idx):
                label = y_cnt[j]
                if label not in [1, 2, 3, 4]:
                    continue  # Only MI tasks (assuming 1-4 labels)

                # For TTA during training
                offsets = [0, int(0.25*fs), int(0.5*fs)] if (is_train and USE_TTA) else [0]

                for off in offsets:
                    s = (start_idx - 1) + off  # matlab indices are 1-based
                    e = s + time_window

                    if e <= X_cnt.shape[0]:
                        raw_epoch = X_cnt[s:e, :22].T  # Transpose to (channels x samples)
                        # Apply bandpass filter
                        filtered_epoch = butter_bandpass_filter(raw_epoch, lowcut=4.0, highcut=38.0, fs=fs, order=4)
                        all_X_epochs.append(filtered_epoch)
                        all_y_labels.append(label - 1)  # Adjust labels to 0-3

    except Exception as e:
        print(f"Error loading {file_path}: {e}")
        return None, None

    if len(all_X_epochs) == 0:
        return None, None

    return np.stack(all_X_epochs).astype('float32'), np.array(all_y_labels).astype('int32')

def load_bcic_subject(path, subject_id, is_train=True):
    """
    Loads data for a single subject, handling the nested .mat file structure.
    """
    file_type = 'T' if is_train else 'E'
    file_path = f"{path}/A{subject_id:02d}{file_type}.mat"
    x_data, y_data = _load_raw_data_from_mat_file(file_path, fs=FS, time_window=TIME_WINDOW, is_train=is_train)

    # Handle alternative naming
    if x_data is None:
        file_path_alt = f"{path}/A0{subject_id}{file_type}.mat"
        x_data, y_data = _load_raw_data_from_mat_file(file_path_alt, fs=FS, time_window=TIME_WINDOW, is_train=is_train)

    return x_data, y_data

# ==================== MODEL DEFINITION ====================
def EEGInception(n_classes, Chans=22, Samples=1000, dropoutRate=0.5):
    input_shape = (Chans, Samples, 1)
    input_layer = Input(shape=input_shape)

    # Parallel Temporal Kernels (Multi-scale feature extraction)
    p1 = Conv2D(8, (1, 128), padding='same', use_bias=False)(input_layer)
    p1 = BatchNormalization()(p1)
    p1 = DepthwiseConv2D((Chans, 1), depth_multiplier=2, use_bias=False, depthwise_constraint=MaxNorm(1.))(p1)
    p1 = BatchNormalization()(p1)
    p1 = Activation('elu')(p1)

    p2 = Conv2D(8, (1, 64), padding='same', use_bias=False)(input_layer)
    p2 = BatchNormalization()(p2)
    p2 = DepthwiseConv2D((Chans, 1), depth_multiplier=2, use_bias=False, depthwise_constraint=MaxNorm(1.))(p2)
    p2 = BatchNormalization()(p2)
    p2 = Activation('elu')(p2)

    p3 = Conv2D(8, (1, 32), padding='same', use_bias=False)(input_layer)
    p3 = BatchNormalization()(p3)
    p3 = DepthwiseConv2D((Chans, 1), depth_multiplier=2, use_bias=False, depthwise_constraint=MaxNorm(1.))(p3)
    p3 = BatchNormalization()(p3)
    p3 = Activation('elu')(p3)

    # Concatenate & Reduction
    merged = Concatenate(axis=-1)([p1, p2, p3])
    pool1 = AveragePooling2D((1, 4))(merged)
    pool1 = Dropout(dropoutRate)(pool1)

    conv2 = Conv2D(16, (1, 32), padding='same', use_bias=False)(pool1)
    conv2 = BatchNormalization()(conv2)
    conv2 = Activation('elu')(conv2)
    pool2 = AveragePooling2D((1, 2))(conv2)
    pool2 = Dropout(dropoutRate)(pool2)

    flatten = Flatten()(pool2)
    dense = Dense(n_classes, kernel_constraint=MaxNorm(0.25))(flatten)
    softmax = Activation('softmax')(dense)

    return Model(inputs=input_layer, outputs=softmax)

# ==================== MAIN LOSO LOOP ====================
print(f"\n{'='*60}")
print(f"STARTING {MODEL_NAME} - LOSO CROSS-SUBJECT EVALUATION")
print(f"{'='*60}")

# Cache all data
data_cache = {}
print("📥 Caching Raw Data...")
all_subjects = list(range(1, 10))

for s in all_subjects:
    X_tr, y_tr = load_bcic_subject(DATA_PATH, s, is_train=True)
    X_te, y_te = load_bcic_subject(DATA_PATH, s, is_train=False)

    if X_tr is not None:
        print(f"   Subject {s}: Train {X_tr.shape}, Test {X_te.shape}")
        data_cache[s] = {'X_tr': X_tr, 'y_tr': y_tr, 'X_te': X_te, 'y_te': y_te}

results = {}
kappa_scores = {}
f1_scores = {}

for target_sub in all_subjects:
    start_time = time.time()
    print(f"\n🎯 TARGET SUBJECT: {target_sub}")

    if target_sub not in data_cache:
        print(f"   ⚠️  No data for subject {target_sub}, skipping...")
        continue

    # Prepare source data (all subjects except target)
    X_source_list, y_source_list = [], []
    for src in all_subjects:
        if src != target_sub and src in data_cache:
            X_source_list.append(data_cache[src]['X_tr'])
            y_source_list.append(data_cache[src]['y_tr'])

    if not X_source_list:
        print(f"   ⚠️  No source data available, skipping...")
        continue

    X_source = np.concatenate(X_source_list, axis=0)
    y_source = np.concatenate(y_source_list, axis=0)

    # Prepare target data
    X_tgt_tr = data_cache[target_sub]['X_tr']
    y_tgt_tr = data_cache[target_sub]['y_tr']
    X_tgt_te = data_cache[target_sub]['X_te']
    y_tgt_te = data_cache[target_sub]['y_te']

    # Apply Euclidean Alignment
    print("   ⚙️ Applying Euclidean Alignment...")
    X_source_aligned = euclidean_alignment(X_source)
    X_tgt_tr_aligned = euclidean_alignment(X_tgt_tr)
    X_tgt_te_aligned = euclidean_alignment(X_tgt_te)

    # Apply Standard Scaling
    print("   ⚗️ Applying Standard Scaling...")
    X_source_final, scaler = scale_data(X_source_aligned, fit=True)
    X_tgt_tr_final, _ = scale_data(X_tgt_tr_aligned, scaler=scaler, fit=False)
    X_tgt_te_final, _ = scale_data(X_tgt_te_aligned, scaler=scaler, fit=False)

    # Add channel dimension for CNN
    X_source_final = np.expand_dims(X_source_final, -1)
    X_tgt_tr_final = np.expand_dims(X_tgt_tr_final, -1)
    X_tgt_te_final = np.expand_dims(X_tgt_te_final, -1)

    # Step 1: Pre-training (General Features)
    print("   🏗️  Initializing EEGInception model...")
    model = EEGInception(n_classes=N_CLASSES)

    print(f"   🚀 Pre-training on source data ({len(y_source)} trials)...")
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])

    # Early stopping callback
    early_stopping = tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True,
        verbose=0
    )

    # Split source data for validation
    from sklearn.model_selection import train_test_split
    X_source_train, X_source_val, y_source_train, y_source_val = train_test_split(
        X_source_final, y_source, test_size=0.2, random_state=42, stratify=y_source
    )

    # Pre-train with validation
    history_pre = model.fit(
        X_source_train, y_source_train,
        validation_data=(X_source_val, y_source_val),
        batch_size=BATCH_SIZE,
        epochs=40,
        verbose=0,
        callbacks=[early_stopping]
    )

    best_pre_acc = max(history_pre.history['val_accuracy']) * 100
    print(f"   ✅ Pre-training completed. Best val acc: {best_pre_acc:.2f}%")

    # Step 2: Fine-tuning (Personalized Adaptation)
    print(f"   🔧 Fine-tuning on Subject {target_sub}...")
    # Slower learning rate to avoid destroying pre-trained filters
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])

    # Split target train for validation during fine-tuning
    X_tgt_tr_final_split, X_tgt_val_final, y_tgt_tr_split, y_tgt_val = train_test_split(
        X_tgt_tr_final, y_tgt_tr, test_size=0.2, random_state=42, stratify=y_tgt_tr
    )

    history_ft = model.fit(
        X_tgt_tr_final_split, y_tgt_tr_split,
        validation_data=(X_tgt_val_final, y_tgt_val),
        batch_size=16,
        epochs=15,
        verbose=0,
        callbacks=[early_stopping]
    )

    best_ft_acc = max(history_ft.history['val_accuracy']) * 100
    print(f"   ✅ Fine-tuning completed. Best val acc: {best_ft_acc:.2f}%")

    # Step 3: Evaluation
    print("   📊 Evaluating on test set...")

    # Get predictions and probabilities
    y_pred_probs = model.predict(X_tgt_te_final, verbose=0)
    y_pred = np.argmax(y_pred_probs, axis=1)
    y_true = y_tgt_te

    # Calculate metrics
    accuracy = 100 * np.mean(y_pred == y_true)
    kappa = cohen_kappa_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average='weighted')

    # Save results with same structure as Code 2
    save_file = os.path.join(SAVE_PATH, f"S{target_sub}")

    # Save probabilities, predictions, and labels
    np.save(f"{save_file}_probs.npy", y_pred_probs)
    np.save(f"{save_file}_preds.npy", y_pred)
    np.save(f"{save_file}_labels.npy", y_true)

    # Save metrics array [accuracy, kappa, f1]
    metrics = np.array([accuracy, kappa, f1])
    np.save(f"{save_file}_metrics.npy", metrics)

    # Store results
    results[target_sub] = accuracy
    kappa_scores[target_sub] = kappa
    f1_scores[target_sub] = f1

    elapsed = time.time() - start_time
    print(f"   📊 Metrics - Acc: {accuracy:.2f}%, Kappa: {kappa:.3f}, F1: {f1:.3f}")
    print(f"   💾 Saved to {save_file}_*.npy (Time: {elapsed:.0f}s)")

    # Clear session to free memory
    tf.keras.backend.clear_session()

# ==================== SAVE OVERALL RESULTS ====================
print(f"\n{'='*50}")
print(f"{MODEL_NAME} FINAL RESULTS")
print(f"{'='*50}")

if results:
    accuracies = list(results.values())
    kappas = list(kappa_scores.values())
    f1s = list(f1_scores.values())

    print(f"{'Subject':<10} | {'Accuracy':<10} | {'Kappa':<8} | {'F1 Score':<8}")
    print("-" * 45)

    for sub in all_subjects:
        if sub in results:
            print(f"{sub:<10} | {results[sub]:.2f}%      | {kappa_scores[sub]:.3f}    | {f1_scores[sub]:.3f}")

    print("-" * 45)
    print(f"MEAN       | {np.mean(accuracies):.2f}%      | {np.mean(kappas):.3f}    | {np.mean(f1s):.3f}")
    print(f"STD DEV    | {np.std(accuracies):.2f}%      | {np.std(kappas):.3f}    | {np.std(f1s):.3f}")

    # Save summary JSON
    summary = {
        'model': MODEL_NAME,
        'accuracies': {str(k): float(v) for k, v in results.items()},
        'kappa_scores': {str(k): float(v) for k, v in kappa_scores.items()},
        'f1_scores': {str(k): float(v) for k, v in f1_scores.items()},
        'mean_accuracy': float(np.mean(accuracies)),
        'std_accuracy': float(np.std(accuracies)),
        'mean_kappa': float(np.mean(kappas)),
        'std_kappa': float(np.std(kappas)),
        'mean_f1': float(np.mean(f1s)),
        'std_f1': float(np.std(f1s)),
        'parameters': {
            'n_classes': N_CLASSES,
            'n_channels': N_CHANNELS,
            'time_window': TIME_WINDOW,
            'fs': FS,
            'data_path': DATA_PATH,
            'save_path': SAVE_PATH
        }
    }

    summary_path = os.path.join(SAVE_PATH, 'summary.json')
    with open(summary_path, 'w') as f:
        json.dump(summary, f, indent=2)

    print(f"\n✅ {MODEL_NAME} training completed!")
    print(f"📁 Results saved to: {SAVE_PATH}/")
    print(f"📄 Summary saved to: {summary_path}")

    # Save numpy arrays with all results
    np.save(os.path.join(SAVE_PATH, 'all_accuracies.npy'), accuracies)
    np.save(os.path.join(SAVE_PATH, 'all_kappas.npy'), kappas)
    np.save(os.path.join(SAVE_PATH, 'all_f1s.npy'), f1s)

else:
    print("❌ No results to save - training failed for all subjects")

print(f"\n{'='*60}")
print(f"ENSEMBLE READY: You can now use the saved outputs in ensemble methods")
print(f"Each subject has: S{target_sub}_probs.npy, S{target_sub}_preds.npy,")
print(f"                  S{target_sub}_labels.npy, S{target_sub}_metrics.npy")
print(f"{'='*60}")


STARTING EEGInception - LOSO CROSS-SUBJECT EVALUATION
📥 Caching Raw Data...
   Subject 1: Train (864, 22, 1000), Test (288, 22, 1000)
   Subject 2: Train (864, 22, 1000), Test (288, 22, 1000)
   Subject 3: Train (864, 22, 1000), Test (288, 22, 1000)
   Subject 4: Train (864, 22, 1000), Test (288, 22, 1000)
   Subject 5: Train (864, 22, 1000), Test (288, 22, 1000)
   Subject 6: Train (864, 22, 1000), Test (288, 22, 1000)
   Subject 7: Train (864, 22, 1000), Test (288, 22, 1000)
   Subject 8: Train (864, 22, 1000), Test (288, 22, 1000)
   Subject 9: Train (864, 22, 1000), Test (288, 22, 1000)

🎯 TARGET SUBJECT: 1
   ⚙️ Applying Euclidean Alignment...
   ... Applying Euclidean Alignment (Shape: (6912, 22, 1000))
   ... Applying Euclidean Alignment (Shape: (864, 22, 1000))
   ... Applying Euclidean Alignment (Shape: (288, 22, 1000))
   ⚗️ Applying Standard Scaling...
   🏗️  Initializing EEGInception model...
   🚀 Pre-training on source data (6912 trials)...
   ✅ Pre-training completed. Be

In [ ]:
# 4

import os
import numpy as np
import scipy.io as sio
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input, Conv2D, BatchNormalization, Activation,
                                     DepthwiseConv2D, SeparableConv2D, AveragePooling2D,
                                     Dropout, Flatten, Dense)
from tensorflow.keras.constraints import max_norm
from scipy.signal import butter, lfilter
import json
from sklearn.metrics import cohen_kappa_score, f1_score, accuracy_score
import time
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# ==================== CONFIG ====================
MODEL_NAME = 'EEGNet_NoAug'
SAVE_PATH = f'/content/gdrive/MyDrive/BCI_ENSEMBLE/{MODEL_NAME}'
DATA_PATH = "/content/gdrive/MyDrive/BCICIV-2a-mat"
FS = 250
N_CLASSES = 4
N_CHANNELS = 22
TIME_WINDOW = 1000
BATCH_SIZE = 64

# EEGNet parameters
DROPOUT_RATE = 0.5
KERNEL_LENGTH = 64  # Increased for better temporal feature extraction
F1 = 8
F2 = 16
D = 2

# Training parameters
LR_PRETRAIN = 0.001
LR_FINETUNE = 0.0001
EPOCHS_PRETRAIN = 80
EPOCHS_FINETUNE = 40

# Create save directory
os.makedirs(SAVE_PATH, exist_ok=True)

# ==================== PREPROCESSING ====================
def butter_bandpass_filter(data, lowcut=4.0, highcut=38.0, fs=250, order=4):
    """Apply bandpass filter to EEG data"""
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    b, a = butter(order, [low, high], btype='band')
    return lfilter(b, a, data, axis=-1)

def apply_common_average_reference(data):
    """Apply Common Average Reference (CAR) filter"""
    avg = np.mean(data, axis=1, keepdims=True)
    return data - avg

def preprocess_pipeline(X):
    """Simple preprocessing pipeline"""
    X_processed = X.copy()

    # 1. Bandpass filter
    X_processed = butter_bandpass_filter(X_processed, lowcut=4.0, highcut=38.0, fs=FS, order=4)

    # 2. CAR filter
    X_processed = apply_common_average_reference(X_processed)

    return X_processed

def euclidean_alignment(X_data, eps=1e-7):
    """Apply Euclidean Alignment"""
    print(f"   ... Applying Euclidean Alignment (Shape: {X_data.shape})")
    # Calculate covariance matrices
    covariances = np.array([np.dot(x, x.T) for x in X_data])
    mean_cov = np.mean(covariances, axis=0)

    # Eigen decomposition with regularization
    d, v = np.linalg.eigh(mean_cov)
    d_reg = np.maximum(d, eps)
    d_inv_sqrt = np.diag(1.0 / np.sqrt(d_reg))
    whitening_mat = np.dot(v, np.dot(d_inv_sqrt, v.T))

    # Apply whitening
    X_aligned = np.array([np.dot(whitening_mat, x) for x in X_data])
    return X_aligned

def scale_data(X, scaler=None, fit=False):
    """Apply standardization to EEG data"""
    n, c, t = X.shape
    x_flat = X.transpose(0, 2, 1).reshape(-1, c)

    if fit:
        scaler = StandardScaler()
        x_scaled = scaler.fit_transform(x_flat)
    else:
        x_scaled = scaler.transform(x_flat)

    x_scaled = x_scaled.reshape(n, t, c).transpose(0, 2, 1)
    return x_scaled, scaler

# ==================== DATA LOADING ====================
def _load_raw_data_from_mat_file(file_path, fs, time_window):
    """Load data from .mat file"""
    all_X_epochs, all_y_labels = [], []
    try:
        mat = sio.loadmat(file_path)
        data_struct = mat['data']

        for i in range(data_struct.shape[1]):
            run_data = data_struct[0][i]
            X_cnt = run_data['X'][0][0]
            trial_idx = run_data['trial'][0][0].flatten()
            y_cnt = run_data['y'][0][0].flatten()

            for j, start_idx in enumerate(trial_idx):
                label = y_cnt[j]
                if label not in [1, 2, 3, 4]:
                    continue

                s = (start_idx - 1)
                e = s + time_window

                if e <= X_cnt.shape[0]:
                    raw_epoch = X_cnt[s:e, :22].T

                    # Apply preprocessing
                    raw_epoch_expanded = np.expand_dims(raw_epoch, axis=0)
                    processed = preprocess_pipeline(raw_epoch_expanded)
                    filtered_epoch = processed[0]

                    all_X_epochs.append(filtered_epoch)
                    all_y_labels.append(label - 1)

    except Exception as e:
        print(f"Error loading {file_path}: {e}")
        return None, None

    if len(all_X_epochs) == 0:
        return None, None

    return np.stack(all_X_epochs).astype('float32'), np.array(all_y_labels).astype('int32')

def load_bcic_subject(path, subject_id, is_train=True):
    """Load subject data"""
    file_type = 'T' if is_train else 'E'
    file_path = f"{path}/A{subject_id:02d}{file_type}.mat"
    x_data, y_data = _load_raw_data_from_mat_file(file_path, fs=FS, time_window=TIME_WINDOW)

    if x_data is None:
        file_path_alt = f"{path}/A0{subject_id}{file_type}.mat"
        x_data, y_data = _load_raw_data_from_mat_file(file_path_alt, fs=FS, time_window=TIME_WINDOW)

    return x_data, y_data

# ==================== EEGNET MODEL ====================
def EEGNet(nb_classes, Chans=22, Samples=1000,
           dropoutRate=DROPOUT_RATE, kernLength=KERNEL_LENGTH,
           F1=F1, D=D, F2=F2, norm_rate=0.25):
    """
    EEGNet Model from https://arxiv.org/abs/1611.08024
    """

    input1 = Input(shape=(Chans, Samples, 1))

    # Block 1: Temporal Convolution
    block1 = Conv2D(F1, (1, kernLength), padding='same',
                   input_shape=(Chans, Samples, 1),
                   use_bias=False)(input1)
    block1 = BatchNormalization()(block1)

    # Depthwise Convolution
    block1 = DepthwiseConv2D((Chans, 1), use_bias=False,
                            depth_multiplier=D,
                            depthwise_constraint=max_norm(1.))(block1)
    block1 = BatchNormalization()(block1)
    block1 = Activation('elu')(block1)

    block1 = AveragePooling2D((1, 4))(block1)
    block1 = Dropout(dropoutRate)(block1)

    # Block 2: Separable Convolution
    block2 = SeparableConv2D(F2, (1, 16),
                            use_bias=False, padding='same')(block1)
    block2 = BatchNormalization()(block2)
    block2 = Activation('elu')(block2)

    block2 = AveragePooling2D((1, 8))(block2)
    block2 = Dropout(dropoutRate)(block2)

    # Classification Block
    flatten = Flatten(name='flatten')(block2)

    dense = Dense(nb_classes, name='dense',
                 kernel_constraint=max_norm(norm_rate))(flatten)
    softmax = Activation('softmax', name='softmax')(dense)

    return Model(inputs=input1, outputs=softmax)

# ==================== TRAINING UTILITIES ====================
def create_class_weights(y):
    """Create class weights for imbalanced data"""
    from sklearn.utils.class_weight import compute_class_weight
    unique = np.unique(y)
    class_weights = compute_class_weight('balanced', classes=unique, y=y)
    return dict(zip(unique, class_weights))

# ==================== MAIN LOSO LOOP ====================
print(f"\n{'='*60}")
print(f"STARTING {MODEL_NAME} - LOSO CROSS-SUBJECT EVALUATION")
print(f"{'='*60}")
print(f"Model: EEGNet (No Augmentation)")
print(f"Parameters: F1={F1}, D={D}, F2={F2}, Kernel Length={KERNEL_LENGTH}")
print(f"Training: Pretrain={EPOCHS_PRETRAIN} epochs, Finetune={EPOCHS_FINETUNE} epochs")
print(f"{'='*60}")

# Cache all data
data_cache = {}
print("📥 Loading Data...")
all_subjects = list(range(1, 10))

for s in all_subjects:
    X_tr, y_tr = load_bcic_subject(DATA_PATH, s, is_train=True)
    X_te, y_te = load_bcic_subject(DATA_PATH, s, is_train=False)

    if X_tr is not None:
        print(f"   Subject {s}: Train {X_tr.shape}, Test {X_te.shape}")
        data_cache[s] = {'X_tr': X_tr, 'y_tr': y_tr, 'X_te': X_te, 'y_te': y_te}
    else:
        print(f"   ⚠️  Could not load data for subject {s}")

results = {}
kappa_scores = {}
f1_scores = {}

for target_sub in all_subjects:
    start_time = time.time()
    print(f"\n🎯 TARGET SUBJECT: {target_sub}")

    if target_sub not in data_cache:
        print(f"   ⚠️  No data for subject {target_sub}, skipping...")
        continue

    # Prepare source data (all subjects except target)
    X_source_list, y_source_list = [], []
    for src in all_subjects:
        if src != target_sub and src in data_cache:
            X_source_list.append(data_cache[src]['X_tr'])
            y_source_list.append(data_cache[src]['y_tr'])

    if not X_source_list:
        print(f"   ⚠️  No source data available, skipping...")
        continue

    X_source = np.concatenate(X_source_list, axis=0)
    y_source = np.concatenate(y_source_list, axis=0)

    # Prepare target data
    X_tgt_tr = data_cache[target_sub]['X_tr']
    y_tgt_tr = data_cache[target_sub]['y_tr']
    X_tgt_te = data_cache[target_sub]['X_te']
    y_tgt_te = data_cache[target_sub]['y_te']

    print(f"   📊 Data shapes:")
    print(f"     Source: {X_source.shape} (trials), {y_source.shape} (labels)")
    print(f"     Target Train: {X_tgt_tr.shape} (trials), {y_tgt_tr.shape} (labels)")
    print(f"     Target Test: {X_tgt_te.shape} (trials), {y_tgt_te.shape} (labels)")

    # Apply Euclidean Alignment
    print("   ⚙️ Applying Euclidean Alignment...")
    X_source_aligned = euclidean_alignment(X_source)
    X_tgt_tr_aligned = euclidean_alignment(X_tgt_tr)
    X_tgt_te_aligned = euclidean_alignment(X_tgt_te)

    # Apply Standard Scaling
    print("   ⚗️ Applying Standard Scaling...")
    X_source_final, scaler = scale_data(X_source_aligned, fit=True)
    X_tgt_tr_final, _ = scale_data(X_tgt_tr_aligned, scaler=scaler, fit=False)
    X_tgt_te_final, _ = scale_data(X_tgt_te_aligned, scaler=scaler, fit=False)

    # Add channel dimension for CNN
    X_source_final = np.expand_dims(X_source_final, -1)
    X_tgt_tr_final = np.expand_dims(X_tgt_tr_final, -1)
    X_tgt_te_final = np.expand_dims(X_tgt_te_final, -1)

    # Create class weights
    class_weights = create_class_weights(y_source)
    print(f"   ⚖️  Class weights: {class_weights}")

    # Step 1: Pre-training (General Features)
    print("   🏗️  Initializing EEGNet model...")
    model = EEGNet(nb_classes=N_CLASSES, Chans=N_CHANNELS, Samples=TIME_WINDOW,
                   dropoutRate=DROPOUT_RATE, kernLength=KERNEL_LENGTH,
                   F1=F1, D=D, F2=F2)

    # Compile model
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LR_PRETRAIN),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    # Callbacks
    early_stopping = tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True,
        verbose=0
    )

    # Split source data for validation
    from sklearn.model_selection import train_test_split
    X_source_train, X_source_val, y_source_train, y_source_val = train_test_split(
        X_source_final, y_source, test_size=0.2, random_state=42, stratify=y_source
    )

    print(f"   🚀 Pre-training on source data ({len(y_source_train)} trials)...")
    history_pre = model.fit(
        X_source_train, y_source_train,
        validation_data=(X_source_val, y_source_val),
        batch_size=BATCH_SIZE,
        epochs=EPOCHS_PRETRAIN,
        verbose=0,
        callbacks=[early_stopping],
        class_weight=class_weights
    )

    best_pre_acc = max(history_pre.history['val_accuracy']) * 100
    print(f"   ✅ Pre-training completed. Best val acc: {best_pre_acc:.2f}%")
    print(f"   📈 Final training acc: {history_pre.history['accuracy'][-1]*100:.2f}%")

    # Step 2: Fine-tuning (Personalized Adaptation)
    print(f"   🔧 Fine-tuning on Subject {target_sub}...")

    # Recompile with lower learning rate
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LR_FINETUNE),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    # Split target train for validation during fine-tuning
    X_tgt_tr_final_split, X_tgt_val_final, y_tgt_tr_split, y_tgt_val = train_test_split(
        X_tgt_tr_final, y_tgt_tr, test_size=0.2, random_state=42, stratify=y_tgt_tr
    )

    # Create class weights for target data
    class_weights_target = create_class_weights(y_tgt_tr_split)

    history_ft = model.fit(
        X_tgt_tr_final_split, y_tgt_tr_split,
        validation_data=(X_tgt_val_final, y_tgt_val),
        batch_size=32,
        epochs=EPOCHS_FINETUNE,
        verbose=0,
        callbacks=[early_stopping],
        class_weight=class_weights_target
    )

    best_ft_acc = max(history_ft.history['val_accuracy']) * 100
    print(f"   ✅ Fine-tuning completed. Best val acc: {best_ft_acc:.2f}%")
    print(f"   📈 Final training acc: {history_ft.history['accuracy'][-1]*100:.2f}%")

    # Step 3: Evaluation
    print("   📊 Evaluating on test set...")

    # Get predictions and probabilities
    y_pred_probs = model.predict(X_tgt_te_final, verbose=0)
    y_pred = np.argmax(y_pred_probs, axis=1)
    y_true = y_tgt_te

    # Calculate metrics
    accuracy = 100 * accuracy_score(y_true, y_pred)
    kappa = cohen_kappa_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average='weighted')

    # Calculate per-class accuracy
    from sklearn.metrics import confusion_matrix
    cm = confusion_matrix(y_true, y_pred)
    per_class_acc = np.diag(cm) / cm.sum(axis=1)

    print(f"   📊 Per-class accuracy: {per_class_acc}")
    print(f"   📊 Confusion matrix:\n{cm}")

    # Save results with same structure as other models
    save_file = os.path.join(SAVE_PATH, f"S{target_sub}")

    # Save all outputs
    np.save(f"{save_file}_probs.npy", y_pred_probs)
    np.save(f"{save_file}_preds.npy", y_pred)
    np.save(f"{save_file}_labels.npy", y_true)

    # Save metrics array [accuracy, kappa, f1]
    metrics = np.array([accuracy, kappa, f1])
    np.save(f"{save_file}_metrics.npy", metrics)

    # Save confusion matrix
    np.save(f"{save_file}_cm.npy", cm)

    # Save training history
    np.save(f"{save_file}_history_pre.npy", history_pre.history)
    np.save(f"{save_file}_history_ft.npy", history_ft.history)

    # Store results
    results[target_sub] = accuracy
    kappa_scores[target_sub] = kappa
    f1_scores[target_sub] = f1

    elapsed = time.time() - start_time
    print(f"   📊 Test Metrics - Acc: {accuracy:.2f}%, Kappa: {kappa:.3f}, F1: {f1:.3f}")
    print(f"   💾 Saved to {save_file}_*.npy (Time: {elapsed:.0f}s)")

    # Clear session to free memory
    tf.keras.backend.clear_session()

# ==================== SAVE OVERALL RESULTS ====================
print(f"\n{'='*60}")
print(f"{MODEL_NAME} FINAL RESULTS")
print(f"{'='*60}")

if results:
    accuracies = list(results.values())
    kappas = list(kappa_scores.values())
    f1s = list(f1_scores.values())

    print(f"\n{'Subject':<10} | {'Accuracy':<10} | {'Kappa':<8} | {'F1 Score':<8}")
    print("-" * 45)

    for sub in sorted(results.keys()):
        print(f"{sub:<10} | {results[sub]:8.2f}%  | {kappa_scores[sub]:7.3f} | {f1_scores[sub]:7.3f}")

    print("-" * 45)
    print(f"{'MEAN':<10} | {np.mean(accuracies):8.2f}%  | {np.mean(kappas):7.3f} | {np.mean(f1s):7.3f}")
    print(f"{'STD DEV':<10} | {np.std(accuracies):8.2f}%  | {np.std(kappas):7.3f} | {np.std(f1s):7.3f}")
    print(f"{'MIN':<10} | {np.min(accuracies):8.2f}%  | {np.min(kappas):7.3f} | {np.min(f1s):7.3f}")
    print(f"{'MAX':<10} | {np.max(accuracies):8.2f}%  | {np.max(kappas):7.3f} | {np.max(f1s):7.3f}")

    # Save comprehensive summary
    summary = {
        'model': MODEL_NAME,
        'model_parameters': {
            'F1': F1,
            'D': D,
            'F2': F2,
            'dropout_rate': DROPOUT_RATE,
            'kernel_length': KERNEL_LENGTH
        },
        'training_parameters': {
            'pretrain_epochs': EPOCHS_PRETRAIN,
            'finetune_epochs': EPOCHS_FINETUNE,
            'pretrain_lr': LR_PRETRAIN,
            'finetune_lr': LR_FINETUNE,
            'batch_size': BATCH_SIZE
        },
        'subject_results': {
            str(k): {
                'accuracy': float(results[k]),
                'kappa': float(kappa_scores[k]),
                'f1': float(f1_scores[k])
            } for k in results.keys()
        },
        'statistics': {
            'mean_accuracy': float(np.mean(accuracies)),
            'std_accuracy': float(np.std(accuracies)),
            'mean_kappa': float(np.mean(kappas)),
            'std_kappa': float(np.std(kappas)),
            'mean_f1': float(np.mean(f1s)),
            'std_f1': float(np.std(f1s))
        },
        'dataset_info': {
            'n_classes': N_CLASSES,
            'n_channels': N_CHANNELS,
            'time_window': TIME_WINDOW,
            'sampling_rate': FS,
            'n_subjects': len(results),
            'data_path': DATA_PATH,
            'save_path': SAVE_PATH,
            'timestamp': time.strftime("%Y-%m-%d %H:%M:%S")
        }
    }

    summary_path = os.path.join(SAVE_PATH, 'summary.json')
    with open(summary_path, 'w') as f:
        json.dump(summary, f, indent=2)

    # Save aggregated results as numpy arrays
    np.save(os.path.join(SAVE_PATH, 'all_accuracies.npy'), accuracies)
    np.save(os.path.join(SAVE_PATH, 'all_kappas.npy'), kappas)
    np.save(os.path.join(SAVE_PATH, 'all_f1s.npy'), f1s)

    # Create a results table file
    with open(os.path.join(SAVE_PATH, 'results_table.txt'), 'w') as f:
        f.write(f"{'='*60}\n")
        f.write(f"{MODEL_NAME} - LOSO Evaluation Results\n")
        f.write(f"{'='*60}\n\n")
        f.write(f"{'Subject':<10} | {'Accuracy':<10} | {'Kappa':<8} | {'F1':<8}\n")
        f.write("-" * 45 + "\n")
        for sub in sorted(results.keys()):
            f.write(f"{sub:<10} | {results[sub]:8.2f}%  | {kappa_scores[sub]:7.3f} | {f1_scores[sub]:7.3f}\n")
        f.write("-" * 45 + "\n")
        f.write(f"{'MEAN':<10} | {np.mean(accuracies):8.2f}%  | {np.mean(kappas):7.3f} | {np.mean(f1s):7.3f}\n")
        f.write(f"{'STD':<10} | {np.std(accuracies):8.2f}%  | {np.std(kappas):7.3f} | {np.std(f1s):7.3f}\n\n")
        f.write(f"Total Subjects: {len(results)}\n")
        f.write(f"Average Accuracy: {np.mean(accuracies):.2f}% ± {np.std(accuracies):.2f}%\n")
        f.write(f"Average Kappa: {np.mean(kappas):.3f} ± {np.std(kappas):.3f}\n")
        f.write(f"Average F1: {np.mean(f1s):.3f} ± {np.std(f1s):.3f}\n")

    print(f"\n✅ {MODEL_NAME} training completed!")
    print(f"📁 Results saved to: {SAVE_PATH}/")
    print(f"📄 Summary saved to: {summary_path}")
    print(f"📋 Results table: {SAVE_PATH}/results_table.txt")

    print(f"\n📊 Performance Summary:")
    print(f"   • Average Accuracy: {np.mean(accuracies):.2f}% ± {np.std(accuracies):.2f}%")
    print(f"   • Average Kappa: {np.mean(kappas):.3f} ± {np.std(kappas):.3f}")
    print(f"   • Average F1 Score: {np.mean(f1s):.3f} ± {np.std(f1s):.3f}")
    print(f"   • Range: {np.min(accuracies):.2f}% - {np.max(accuracies):.2f}%")

    # Calculate and display confidence intervals (95%)
    from scipy import stats
    if len(accuracies) > 1:
        t_critical = stats.t.ppf(q=0.975, df=len(accuracies)-1)
        margin_error = t_critical * (np.std(accuracies) / np.sqrt(len(accuracies)))
        ci_lower = np.mean(accuracies) - margin_error
        ci_upper = np.mean(accuracies) + margin_error
        print(f"   • 95% CI for Accuracy: [{ci_lower:.2f}%, {ci_upper:.2f}%]")

else:
    print("❌ No results to save - training failed for all subjects")

print(f"\n{'='*60}")
print(f"ENSEMBLE READY: Outputs compatible with other models")
print(f"Each subject folder contains:")
print(f"  • S{target_sub}_probs.npy     - Class probabilities")
print(f"  • S{target_sub}_preds.npy     - Predicted labels")
print(f"  • S{target_sub}_labels.npy    - True labels")
print(f"  • S{target_sub}_metrics.npy   - [accuracy, kappa, f1]")
print(f"  • S{target_sub}_cm.npy        - Confusion matrix")
print(f"  • S{target_sub}_history_pre.npy - Pre-training history")
print(f"  • S{target_sub}_history_ft.npy  - Fine-tuning history")
print(f"{'='*60}")
print(f"\nThese outputs can be directly used in ensemble methods alongside:")
print(f"  • Transformer model outputs")
print(f"  • EEGInception model outputs")
print(f"{'='*60}")


STARTING EEGNet_NoAug - LOSO CROSS-SUBJECT EVALUATION
Model: EEGNet (No Augmentation)
Parameters: F1=8, D=2, F2=16, Kernel Length=64
Training: Pretrain=80 epochs, Finetune=40 epochs
📥 Loading Data...
   Subject 1: Train (288, 22, 1000), Test (288, 22, 1000)
   Subject 2: Train (288, 22, 1000), Test (288, 22, 1000)
   Subject 3: Train (288, 22, 1000), Test (288, 22, 1000)
   Subject 4: Train (288, 22, 1000), Test (288, 22, 1000)
   Subject 5: Train (288, 22, 1000), Test (288, 22, 1000)
   Subject 6: Train (288, 22, 1000), Test (288, 22, 1000)
   Subject 7: Train (288, 22, 1000), Test (288, 22, 1000)
   Subject 8: Train (288, 22, 1000), Test (288, 22, 1000)
   Subject 9: Train (288, 22, 1000), Test (288, 22, 1000)

🎯 TARGET SUBJECT: 1
   📊 Data shapes:
     Source: (2304, 22, 1000) (trials), (2304,) (labels)
     Target Train: (288, 22, 1000) (trials), (288,) (labels)
     Target Test: (288, 22, 1000) (trials), (288,) (labels)
   ⚙️ Applying Euclidean Alignment...
   ... Applying Euclid

In [ ]:
# ==========================================
# CELL 5: EEG-SWIN TRANSFORMER MODEL
# ==========================================
import os
import numpy as np
import scipy.io
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from scipy.signal import butter, lfilter
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import cohen_kappa_score, f1_score
import time
import json
import math

# ==================== CONFIG ====================
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Device: {DEVICE}")

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

FS = 250
N_CLASSES = 4
N_CHANNELS = 22
TIME_WINDOW = 1000
USE_TTA = True
LABEL_SMOOTHING = 0.1
BATCH_SIZE = 32
LR_PRETRAIN = 0.0005
LR_FINETUNE = 0.0001
EPOCHS_PRETRAIN = 200
EPOCHS_FINETUNE = 50

DATA_PATH = '/content/gdrive/MyDrive/BCICIV-2a-mat'
MODEL_NAME = 'EEG_Swin'
SAVE_PATH = f'/content/gdrive/MyDrive/BCI_ENSEMBLE/{MODEL_NAME}'

# ==================== UTILS (Same as before) ====================
def butter_bandpass_filter(data, lowcut=4.0, highcut=38.0, fs=250, order=5):
    nyq = 0.5 * fs
    b, a = butter(order, [lowcut/nyq, highcut/nyq], btype='band')
    return lfilter(b, a, data, axis=-1)

def euclidean_alignment(X_data):
    print(f"   ... Applying Euclidean Alignment (Shape: {X_data.shape})")
    covariances = np.matmul(X_data, np.transpose(X_data, (0, 2, 1)))
    mean_cov = np.mean(covariances, axis=0)
    d, v = np.linalg.eigh(mean_cov)
    d_inv_sqrt = np.diag(1.0 / np.sqrt(d + 1e-7))
    whitening_mat = np.dot(v, np.dot(d_inv_sqrt, v.T))
    X_transposed = np.transpose(X_data, (0, 2, 1))
    X_aligned = np.matmul(X_transposed, whitening_mat)
    return np.transpose(X_aligned, (0, 2, 1))

def scale_data(X, scaler, fit=False):
    n, c, t = X.shape
    x_flat = X.transpose(0, 2, 1).reshape(-1, c)
    if fit:
        x_scaled = scaler.fit_transform(x_flat)
    else:
        x_scaled = scaler.transform(x_flat)
    return x_scaled.reshape(n, t, c).transpose(0, 2, 1)

def load_bci_data_raw(subject_id, base_path, is_train=True):
    file_type = 'T' if is_train else 'E'
    file_name = f"A{subject_id:02d}{file_type}.mat"
    full_path = os.path.join(base_path, file_name)

    if not os.path.exists(full_path):
        file_name = f"A0{subject_id}{file_type}.mat"
        full_path = os.path.join(base_path, file_name)
        if not os.path.exists(full_path): return None, None

    try:
        mat = scipy.io.loadmat(full_path)
        data_struct = mat['data']
        all_X, all_y = [], []

        for i in range(data_struct.shape[1]):
            try:
                run_data = data_struct[0][i]
                X_cnt = run_data['X'][0][0]
                trial_idx = run_data['trial'][0][0].flatten()
                y_cnt = run_data['y'][0][0].flatten()

                if len(trial_idx) == 0: continue

                for j, start_idx in enumerate(trial_idx):
                    label = y_cnt[j]
                    if label not in [1, 2, 3, 4]: continue

                    offsets = [0, int(0.25*FS), int(0.5*FS)] if (is_train and USE_TTA) else [0]

                    for off in offsets:
                        s = (start_idx - 1) + off
                        e = s + TIME_WINDOW

                        if e <= X_cnt.shape[0]:
                            raw_epoch = X_cnt[s:e, :22].T
                            filtered = butter_bandpass_filter(raw_epoch, 4.0, 38.0, FS, 4)
                            all_X.append(filtered)
                            all_y.append(label - 1)
            except: continue

        if len(all_X) == 0: return None, None
        return np.stack(all_X), np.array(all_y)
    except: return None, None

def get_dataloader(X, y, batch_size, shuffle=True):
    tensor_x = torch.Tensor(X).unsqueeze(1)
    tensor_y = torch.LongTensor(y)
    return DataLoader(TensorDataset(tensor_x, tensor_y), batch_size=batch_size, shuffle=shuffle)

def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for inputs, labels in loader:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    return total_loss / len(loader), 100 * correct / total

def evaluate_probs(model, loader):
    model.eval()
    all_probs = []
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for inputs, labels in loader:
            inputs = inputs.to(DEVICE)
            outputs = model(inputs)
            probs = torch.softmax(outputs, dim=1)
            _, preds = torch.max(outputs, 1)

            all_probs.append(probs.cpu().numpy())
            all_preds.append(preds.cpu().numpy())
            all_labels.append(labels.numpy())

    return (np.vstack(all_probs),
            np.concatenate(all_preds),
            np.concatenate(all_labels))

# ==================== EEG-SWIN TRANSFORMER ====================
class PatchMerging(nn.Module):
    """Patch Merging Layer for Swin Transformer"""
    def __init__(self, dim, norm_layer=nn.LayerNorm):
        super(PatchMerging, self).__init__()
        self.dim = dim
        self.reduction = nn.Linear(4 * dim, 2 * dim, bias=False)
        self.norm = norm_layer(4 * dim)

    def forward(self, x, H, W):
        """
        x: B, H*W, C
        """
        B, L, C = x.shape
        assert L == H * W, "input feature has wrong size"

        x = x.view(B, H, W, C)

        # Merge patches in 2x2 neighborhoods
        x0 = x[:, 0::2, 0::2, :]  # B, H/2, W/2, C
        x1 = x[:, 1::2, 0::2, :]  # B, H/2, W/2, C
        x2 = x[:, 0::2, 1::2, :]  # B, H/2, W/2, C
        x3 = x[:, 1::2, 1::2, :]  # B, H/2, W/2, C

        x = torch.cat([x0, x1, x2, x3], -1)  # B, H/2, W/2, 4*C
        x = x.view(B, -1, 4 * C)  # B, H/2*W/2, 4*C

        x = self.norm(x)
        x = self.reduction(x)

        return x

class WindowAttention(nn.Module):
    """Window based multi-head self attention with relative position bias"""
    def __init__(self, dim, window_size, num_heads, qkv_bias=True, attn_drop=0., proj_drop=0.):
        super(WindowAttention, self).__init__()
        self.dim = dim
        self.window_size = window_size
        self.num_heads = num_heads
        head_dim = dim // num_heads
        self.scale = head_dim ** -0.5

        # Relative position bias
        self.relative_position_bias_table = nn.Parameter(
            torch.zeros((2 * window_size[0] - 1) * (2 * window_size[1] - 1), num_heads)
        )

        coords_h = torch.arange(self.window_size[0])
        coords_w = torch.arange(self.window_size[1])
        coords = torch.stack(torch.meshgrid([coords_h, coords_w]))  # 2, Wh, Ww
        coords_flatten = torch.flatten(coords, 1)  # 2, Wh*Ww
        relative_coords = coords_flatten[:, :, None] - coords_flatten[:, None, :]  # 2, Wh*Ww, Wh*Ww
        relative_coords = relative_coords.permute(1, 2, 0).contiguous()  # Wh*Ww, Wh*Ww, 2
        relative_coords[:, :, 0] += self.window_size[0] - 1  # shift to start from 0
        relative_coords[:, :, 1] += self.window_size[1] - 1
        relative_coords[:, :, 0] *= 2 * self.window_size[1] - 1
        relative_position_index = relative_coords.sum(-1)  # Wh*Ww, Wh*Ww
        self.register_buffer("relative_position_index", relative_position_index)

        self.qkv = nn.Linear(dim, dim * 3, bias=qkv_bias)
        self.attn_drop = nn.Dropout(attn_drop)
        self.proj = nn.Linear(dim, dim)
        self.proj_drop = nn.Dropout(proj_drop)

        nn.init.trunc_normal_(self.relative_position_bias_table, std=.02)
        self.softmax = nn.Softmax(dim=-1)

    def forward(self, x, mask=None):
        B_, N, C = x.shape
        qkv = self.qkv(x).reshape(B_, N, 3, self.num_heads, C // self.num_heads).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]

        q = q * self.scale
        attn = (q @ k.transpose(-2, -1))

        relative_position_bias = self.relative_position_bias_table[self.relative_position_index.view(-1)].view(
            self.window_size[0] * self.window_size[1], self.window_size[0] * self.window_size[1], -1)
        relative_position_bias = relative_position_bias.permute(2, 0, 1).contiguous()
        attn = attn + relative_position_bias.unsqueeze(0)

        if mask is not None:
            nW = mask.shape[0]
            attn = attn.view(B_ // nW, nW, self.num_heads, N, N) + mask.unsqueeze(1).unsqueeze(0)
            attn = attn.view(-1, self.num_heads, N, N)
            attn = self.softmax(attn)
        else:
            attn = self.softmax(attn)

        attn = self.attn_drop(attn)

        x = (attn @ v).transpose(1, 2).reshape(B_, N, C)
        x = self.proj(x)
        x = self.proj_drop(x)
        return x

class SwinTransformerBlock(nn.Module):
    """Swin Transformer Block"""
    def __init__(self, dim, num_heads, window_size=7, shift_size=0,
                 mlp_ratio=4., qkv_bias=True, drop=0., attn_drop=0.,
                 drop_path=0., norm_layer=nn.LayerNorm):
        super(SwinTransformerBlock, self).__init__()
        self.dim = dim
        self.num_heads = num_heads
        self.window_size = window_size
        self.shift_size = shift_size
        self.mlp_ratio = mlp_ratio

        assert 0 <= self.shift_size < self.window_size, "shift_size must be 0 <= shift_size < window_size"

        self.norm1 = norm_layer(dim)
        self.attn = WindowAttention(
            dim, window_size=(self.window_size, self.window_size), num_heads=num_heads,
            qkv_bias=qkv_bias, attn_drop=attn_drop, proj_drop=drop
        )

        self.drop_path = nn.Dropout(drop_path) if drop_path > 0. else nn.Identity()
        self.norm2 = norm_layer(dim)
        mlp_hidden_dim = int(dim * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(dim, mlp_hidden_dim),
            nn.GELU(),
            nn.Dropout(drop),
            nn.Linear(mlp_hidden_dim, dim),
            nn.Dropout(drop)
        )

    def create_mask(self, H, W):
        # Calculate attention mask for shifted window attention
        img_mask = torch.zeros((1, H, W, 1))
        h_slices = (slice(0, -self.window_size),
                    slice(-self.window_size, -self.shift_size),
                    slice(-self.shift_size, None))
        w_slices = (slice(0, -self.window_size),
                    slice(-self.window_size, -self.shift_size),
                    slice(-self.shift_size, None))
        cnt = 0
        for h in h_slices:
            for w in w_slices:
                img_mask[:, h, w, :] = cnt
                cnt += 1

        mask_windows = img_mask.view(1, H // self.window_size, self.window_size,
                                     W // self.window_size, self.window_size, 1)
        mask_windows = mask_windows.permute(0, 1, 3, 2, 4, 5).contiguous()
        mask_windows = mask_windows.view(-1, self.window_size * self.window_size)

        attn_mask = mask_windows.unsqueeze(1) - mask_windows.unsqueeze(2)
        attn_mask = attn_mask.masked_fill(attn_mask != 0, float(-100.0)).masked_fill(attn_mask == 0, float(0.0))

        return attn_mask

    def forward(self, x, H, W):
        B, L, C = x.shape
        assert L == H * W, "input feature has wrong size"

        shortcut = x
        x = self.norm1(x)
        x = x.view(B, H, W, C)

        # Shift window
        if self.shift_size > 0:
            shifted_x = torch.roll(x, shifts=(-self.shift_size, -self.shift_size), dims=(1, 2))
            attn_mask = self.create_mask(H, W).to(x.device)
        else:
            shifted_x = x
            attn_mask = None

        # Partition windows
        x_windows = shifted_x.view(B, H // self.window_size, self.window_size,
                                   W // self.window_size, self.window_size, C)
        x_windows = x_windows.permute(0, 1, 3, 2, 4, 5).contiguous()
        x_windows = x_windows.view(-1, self.window_size * self.window_size, C)

        # Window attention
        attn_windows = self.attn(x_windows, mask=attn_mask)

        # Merge windows
        attn_windows = attn_windows.view(-1, self.window_size, self.window_size, C)
        shifted_x = attn_windows.permute(0, 3, 1, 2).contiguous()
        shifted_x = shifted_x.view(B, H // self.window_size, W // self.window_size,
                                   self.window_size, self.window_size, C)
        shifted_x = shifted_x.permute(0, 1, 4, 2, 5, 3).contiguous()
        shifted_x = shifted_x.view(B, H, W, C)

        # Reverse shift
        if self.shift_size > 0:
            x = torch.roll(shifted_x, shifts=(self.shift_size, self.shift_size), dims=(1, 2))
        else:
            x = shifted_x

        x = x.view(B, H * W, C)
        x = shortcut + self.drop_path(x)

        # FFN
        x = x + self.drop_path(self.mlp(self.norm2(x)))

        return x

class BasicLayer(nn.Module):
    """A basic Swin Transformer layer for one stage"""
    def __init__(self, dim, depth, num_heads, window_size,
                 mlp_ratio=4., qkv_bias=True, drop=0., attn_drop=0.,
                 drop_path=0., norm_layer=nn.LayerNorm, downsample=None):
        super(BasicLayer, self).__init__()
        self.dim = dim
        self.depth = depth
        self.downsample = downsample

        # Build blocks
        self.blocks = nn.ModuleList([
            SwinTransformerBlock(
                dim=dim, num_heads=num_heads, window_size=window_size,
                shift_size=0 if (i % 2 == 0) else window_size // 2,
                mlp_ratio=mlp_ratio, qkv_bias=qkv_bias,
                drop=drop, attn_drop=attn_drop,
                drop_path=drop_path[i] if isinstance(drop_path, list) else drop_path,
                norm_layer=norm_layer
            )
            for i in range(depth)
        ])

    def forward(self, x, H, W):
        # Downsample if needed
        if self.downsample is not None:
            x = self.downsample(x, H, W)
            H, W = H // 2, W // 2

        for blk in self.blocks:
            x = blk(x, H, W)

        return x, H, W

class PatchEmbed(nn.Module):
    """EEG to Patch Embedding"""
    def __init__(self, in_channels=22, patch_size=50, embed_dim=128):
        super(PatchEmbed, self).__init__()
        self.patch_size = patch_size
        self.embed_dim = embed_dim

        # Use 1D convolution along time dimension
        self.proj = nn.Conv2d(1, embed_dim, kernel_size=(in_channels, patch_size),
                             stride=(1, patch_size))
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, x):
        # x: (B, 1, C, T)
        x = self.proj(x)  # (B, embed_dim, 1, T/patch_size)
        x = x.flatten(2).transpose(1, 2)  # (B, num_patches, embed_dim)
        x = self.norm(x)
        return x

class EEGSwinTransformer(nn.Module):
    """
    EEG-Swin Transformer: Adapting Swin Transformer for EEG Classification
    Based on: "Swin Transformer: Hierarchical Vision Transformer using Shifted Windows"
    """
    def __init__(self, n_classes=4, n_channels=22, n_time=1000,
                 embed_dim=128, depths=[2, 2, 6, 2], num_heads=[4, 8, 16, 32],
                 window_size=7, mlp_ratio=4., qkv_bias=True,
                 drop_rate=0., attn_drop_rate=0., drop_path_rate=0.1,
                 norm_layer=nn.LayerNorm, patch_size=50):
        super(EEGSwinTransformer, self).__init__()

        self.n_classes = n_classes
        self.embed_dim = embed_dim
        self.patch_size = patch_size
        self.num_layers = len(depths)

        # Patch partition and linear embedding
        self.patch_embed = PatchEmbed(
            in_channels=n_channels,
            patch_size=patch_size,
            embed_dim=embed_dim
        )

        num_patches = n_time // patch_size

        # Absolute position embedding
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches, embed_dim))
        self.pos_drop = nn.Dropout(p=drop_rate)

        # Stochastic depth
        dpr = [x.item() for x in torch.linspace(0, drop_path_rate, sum(depths))]

        # Build layers
        self.layers = nn.ModuleList()
        for i_layer in range(self.num_layers):
            layer = BasicLayer(
                dim=int(embed_dim * 2 ** i_layer),
                depth=depths[i_layer],
                num_heads=num_heads[i_layer],
                window_size=window_size,
                mlp_ratio=mlp_ratio,
                qkv_bias=qkv_bias,
                drop=drop_rate,
                attn_drop=attn_drop_rate,
                drop_path=dpr[sum(depths[:i_layer]):sum(depths[:i_layer + 1])],
                norm_layer=norm_layer,
                downsample=PatchMerging if (i_layer < self.num_layers - 1) else None
            )
            self.layers.append(layer)

        self.norm = norm_layer(int(embed_dim * 2 ** (self.num_layers - 1)))

        # Classification head
        self.avgpool = nn.AdaptiveAvgPool1d(1)
        self.head = nn.Sequential(
            nn.Linear(int(embed_dim * 2 ** (self.num_layers - 1)), 512),
            nn.GELU(),
            nn.Dropout(0.5),
            nn.Linear(512, 256),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(256, n_classes)
        )

        self._init_weights()

        print(f"   EEG-Swin Transformer Configuration:")
        print(f"     - Embed Dim: {embed_dim}")
        print(f"     - Depths: {depths}")
        print(f"     - Num Heads: {num_heads}")
        print(f"     - Window Size: {window_size}")
        print(f"     - Num Patches: {num_patches}")

    def _init_weights(self):
        nn.init.trunc_normal_(self.pos_embed, std=.02)

    def forward_features(self, x):
        # Patch embedding
        x = self.patch_embed(x)  # (B, num_patches, embed_dim)

        # Add position embedding
        x = x + self.pos_embed
        x = self.pos_drop(x)

        # Initial H, W (for EEG: H=1, W=num_patches)
        H, W = 1, x.shape[1]

        # Forward through layers
        for layer in self.layers:
            x, H, W = layer(x, H, W)

        x = self.norm(x)  # (B, L, C)
        return x

    def forward(self, x):
        x = self.forward_features(x)

        # Global average pooling
        x = x.transpose(1, 2)  # (B, C, L)
        x = self.avgpool(x)    # (B, C, 1)
        x = torch.flatten(x, 1)  # (B, C)

        # Classification
        x = self.head(x)
        return x

# ==================== SIMPLIFIED SWIN ====================
class SimpleSwinTransformer(nn.Module):
    """Simplified Swin Transformer for EEG"""
    def __init__(self, n_classes=4, n_channels=22, n_time=1000):
        super(SimpleSwinTransformer, self).__init__()

        self.patch_size = 50
        self.num_patches = n_time // self.patch_size
        embed_dim = 128

        # Patch embedding
        self.patch_embed = nn.Sequential(
            nn.Conv2d(1, embed_dim, kernel_size=(n_channels, self.patch_size),
                     stride=(1, self.patch_size)),
            nn.Flatten(2),
            nn.LayerNorm(embed_dim)
        )

        # Position embedding
        self.pos_embed = nn.Parameter(torch.randn(1, self.num_patches, embed_dim))

        # Transformer layers with window attention
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=8,
            dim_feedforward=512,
            dropout=0.1,
            activation='gelu',
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=4)

        # Classification head
        self.norm = nn.LayerNorm(embed_dim)
        self.classifier = nn.Sequential(
            nn.Linear(embed_dim, 64),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(64, n_classes)
        )

    def forward(self, x):
        # Patch embedding
        x = self.patch_embed(x)  # (B, embed_dim, num_patches)
        x = x.transpose(1, 2)    # (B, num_patches, embed_dim)

        # Add position embedding
        x = x + self.pos_embed

        # Transformer
        x = self.transformer(x)

        # Global average pooling and classification
        x = self.norm(x)
        x = x.mean(dim=1)  # (B, embed_dim)
        return self.classifier(x)

# ==================== HYBRID SWIN-CNN ====================
class HybridSwinCNN(nn.Module):
    """Hybrid CNN + Swin Transformer"""
    def __init__(self, n_classes=4, n_channels=22, n_time=1000):
        super(HybridSwinCNN, self).__init__()

        # CNN feature extractor
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=(1, 64), padding=(0, 32)),
            nn.BatchNorm2d(32),
            nn.GELU(),
            nn.AvgPool2d((1, 4)),
            nn.Dropout(0.2),

            nn.Conv2d(32, 64, kernel_size=(n_channels, 1)),
            nn.BatchNorm2d(64),
            nn.GELU(),
            nn.AvgPool2d((1, 4)),
            nn.Dropout(0.2),

            nn.Conv2d(64, 128, kernel_size=(1, 16), padding=(0, 8)),
            nn.BatchNorm2d(128),
            nn.GELU(),
            nn.AvgPool2d((1, 2)),
            nn.Dropout(0.2)
        )

        # Calculate CNN output dimensions
        with torch.no_grad():
            dummy = torch.zeros(1, 1, n_channels, n_time)
            cnn_out = self.cnn(dummy)
            cnn_time = cnn_out.size(3)

        # Swin Transformer
        self.patch_embed = nn.Linear(128, 128)
        self.pos_embed = nn.Parameter(torch.randn(1, cnn_time, 128))

        # Simplified Swin blocks
        self.swin_blocks = nn.ModuleList([
            nn.TransformerEncoderLayer(
                d_model=128,
                nhead=8,
                dim_feedforward=256,
                dropout=0.1,
                activation='gelu',
                batch_first=True
            )
            for _ in range(3)
        ])

        # Classification head
        self.norm = nn.LayerNorm(128)
        self.classifier = nn.Sequential(
            nn.Linear(128, 64),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(64, n_classes)
        )

    def forward(self, x):
        # CNN features
        x = self.cnn(x)  # (B, 128, 1, T')
        x = x.squeeze(2)  # (B, 128, T')
        x = x.transpose(1, 2)  # (B, T', 128)

        # Swin Transformer
        x = self.patch_embed(x)
        x = x + self.pos_embed

        for block in self.swin_blocks:
            x = block(x)

        # Global pooling and classification
        x = self.norm(x)
        x = x.mean(dim=1)  # (B, 128)
        return self.classifier(x)

# ==================== MAIN LOSO LOOP ====================
print(f"\n{'='*60}")
print(f"STARTING {MODEL_NAME} - LOSO CROSS-SUBJECT EVALUATION")
print(f"{'='*60}")

# Cache all data
data_cache = {}
print("📥 Caching Raw Data...")
all_subjects = list(range(1, 10))

for s in all_subjects:
    X_tr, y_tr = load_bci_data_raw(s, DATA_PATH, True)
    X_te, y_te = load_bci_data_raw(s, DATA_PATH, False)

    if X_tr is not None:
        print(f"   Subject {s}: Train {X_tr.shape}, Test {X_te.shape}")
        data_cache[s] = {'X_tr': X_tr, 'y_tr': y_tr, 'X_te': X_te, 'y_te': y_te}

results = {}
kappa_scores = {}

for target_sub in all_subjects:
    start_time = time.time()
    print(f"\n🎯 TARGET SUBJECT: {target_sub}")

    # Prepare source data
    X_source_list, y_source_list = [], []
    for src in all_subjects:
        if src != target_sub and src in data_cache:
            X_source_list.append(data_cache[src]['X_tr'])
            y_source_list.append(data_cache[src]['y_tr'])

    if not X_source_list: continue

    X_source = np.concatenate(X_source_list)
    y_source = np.concatenate(y_source_list)

    # Prepare target data
    if target_sub not in data_cache: continue
    X_tgt_tr = data_cache[target_sub]['X_tr']
    y_tgt_tr = data_cache[target_sub]['y_tr']
    X_tgt_te = data_cache[target_sub]['X_te']
    y_tgt_te = data_cache[target_sub]['y_te']

    # Apply Euclidean Alignment
    print("   ⚙️ Applying Euclidean Alignment...")
    X_source_aligned = euclidean_alignment(X_source)
    X_tgt_tr_aligned = euclidean_alignment(X_tgt_tr)
    X_tgt_te_aligned = euclidean_alignment(X_tgt_te)

    # Apply Standard Scaling
    print("   ⚗️ Applying Standard Scaling...")
    scaler = StandardScaler()
    X_source_final = scale_data(X_source_aligned, scaler, fit=True)
    X_tgt_tr_final = scale_data(X_tgt_tr_aligned, scaler, fit=False)
    X_tgt_te_final = scale_data(X_tgt_te_aligned, scaler, fit=False)

    # Create dataloaders
    source_loader = get_dataloader(X_source_final, y_source, BATCH_SIZE)
    tgt_train_loader = get_dataloader(X_tgt_tr_final, y_tgt_tr, BATCH_SIZE)
    tgt_test_loader = get_dataloader(X_tgt_te_final, y_tgt_te, BATCH_SIZE, shuffle=False)

    # Initialize model
    print(f"   🏗️  Initializing {MODEL_NAME} model...")

    # Use HybridSwinCNN for stability
    model = HybridSwinCNN(
        n_classes=N_CLASSES,
        n_channels=N_CHANNELS,
        n_time=TIME_WINDOW
    ).to(DEVICE)

    criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)

    # Print model info
    total_params = sum(p.numel() for p in model.parameters())
    print(f"   📊 Model Parameters: {total_params:,}")
    print(f"   🎯 Model Architecture: HybridSwinCNN")

    # Test forward pass
    print("   🔍 Testing forward pass...")
    try:
        test_input = torch.randn(2, 1, N_CHANNELS, TIME_WINDOW).to(DEVICE)
        test_output = model(test_input)
        print(f"      ✓ Forward pass successful. Output shape: {test_output.shape}")

        # Try full EEGSwinTransformer if hybrid works
        print("   🔄 Testing full EEG-Swin Transformer...")
        full_model = EEGSwinTransformer(
            n_classes=N_CLASSES,
            n_channels=N_CHANNELS,
            n_time=TIME_WINDOW,
            embed_dim=128,
            depths=[2, 2, 6, 2],
            num_heads=[4, 8, 16, 32],
            window_size=7
        ).to(DEVICE)
        full_model(test_input)
        print("      ✓ Full EEG-Swin works. Using full model for training.")
        model = full_model

    except Exception as e:
        print(f"      ⚠️  Using hybrid version: {e}")

    # Pre-training on source
    print(f"   🚀 Pre-training ({EPOCHS_PRETRAIN} epochs)...")
    optimizer_pre = optim.AdamW(model.parameters(), lr=LR_PRETRAIN, weight_decay=0.01)

    best_pre_acc = 0
    for ep in range(EPOCHS_PRETRAIN):
        try:
            loss, acc = train_epoch(model, source_loader, optimizer_pre, criterion)
            best_pre_acc = max(best_pre_acc, acc)
            if (ep+1) % 20 == 0:
                print(f"      Ep {ep+1}: Loss {loss:.4f} | Acc {acc:.2f}% | Best {best_pre_acc:.2f}%")
        except Exception as e:
            print(f"      ⚠️  Error at epoch {ep+1}: {e}")
            # Reduce learning rate
            for param_group in optimizer_pre.param_groups:
                param_group['lr'] *= 0.7
            continue

    # Fine-tuning on target
    print(f"   🔧 Fine-tuning ({EPOCHS_FINETUNE} epochs)...")
    optimizer_ft = optim.AdamW(model.parameters(), lr=LR_FINETUNE, weight_decay=0.01)

    best_ft_acc = 0
    for ep in range(EPOCHS_FINETUNE):
        try:
            loss, acc = train_epoch(model, tgt_train_loader, optimizer_ft, criterion)
            best_ft_acc = max(best_ft_acc, acc)
            if (ep+1) % 5 == 0:
                print(f"      Ep {ep+1}: Loss {loss:.4f} | Acc {acc:.2f}% | Best {best_ft_acc:.2f}%")
        except Exception as e:
            print(f"      ⚠️  Minor error at epoch {ep+1}: {e}, continuing...")
            continue

    # Evaluation
    try:
        probs, preds, labels = evaluate_probs(model, tgt_test_loader)
        accuracy = 100 * (preds == labels).mean()
        kappa = cohen_kappa_score(labels, preds)
        f1 = f1_score(labels, preds, average='weighted')

        # Save results
        save_file = os.path.join(SAVE_PATH, f"S{target_sub}")
        np.save(f"{save_file}_probs.npy", probs)
        np.save(f"{save_file}_labels.npy", labels)
        np.save(f"{save_file}_preds.npy", preds)

        metrics = np.array([accuracy, kappa, f1])
        np.save(f"{save_file}_metrics.npy", metrics)

        results[target_sub] = accuracy
        kappa_scores[target_sub] = kappa

        elapsed = time.time() - start_time
        print(f"   📊 Metrics - Acc: {accuracy:.2f}%, Kappa: {kappa:.3f}, F1: {f1:.3f}")
        print(f"   💾 Saved to {save_file}_*.npy (Time: {elapsed:.0f}s)")

    except Exception as e:
        print(f"   ✗ Evaluation failed: {e}")
        continue

# ==================== SAVE OVERALL RESULTS ====================
print(f"\n{'='*50}")
print(f"{MODEL_NAME} FINAL RESULTS")
print(f"{'='*50}")

if results:
    accuracies = list(results.values())
    print(f"{'Subject':<10} | {'Accuracy':<10} | {'Kappa':<8}")
    print("-" * 35)
    for sub in all_subjects:
        if sub in results:
            print(f"{sub:<10} | {results[sub]:.2f}%      | {kappa_scores[sub]:.3f}")

    print("-" * 35)
    print(f"AVERAGE    | {np.mean(accuracies):.2f}%      | {np.mean(list(kappa_scores.values())):.3f}")
    print(f"STD DEV    | {np.std(accuracies):.2f}%      | {np.std(list(kappa_scores.values())):.3f}")

    # Save summary
    summary = {
        'model': MODEL_NAME,
        'accuracies': results,
        'kappa_scores': kappa_scores,
        'mean_accuracy': float(np.mean(accuracies)),
        'std_accuracy': float(np.std(accuracies)),
        'mean_kappa': float(np.mean(list(kappa_scores.values()))),
        'std_kappa': float(np.std(list(kappa_scores.values())))
    }

    with open(os.path.join(SAVE_PATH, 'summary.json'), 'w') as f:
        json.dump(summary, f, indent=2)

    print(f"\n✅ {MODEL_NAME} training completed! Results saved to {SAVE_PATH}/")
else:
    print("❌ No results to save - training failed for all subjects")

✅ Device: cuda

STARTING EEG_Swin - LOSO CROSS-SUBJECT EVALUATION
📥 Caching Raw Data...
   Subject 1: Train (864, 22, 1000), Test (288, 22, 1000)
   Subject 2: Train (864, 22, 1000), Test (288, 22, 1000)
   Subject 3: Train (864, 22, 1000), Test (288, 22, 1000)
   Subject 4: Train (864, 22, 1000), Test (288, 22, 1000)
   Subject 5: Train (864, 22, 1000), Test (288, 22, 1000)
   Subject 6: Train (864, 22, 1000), Test (288, 22, 1000)
   Subject 7: Train (864, 22, 1000), Test (288, 22, 1000)
   Subject 8: Train (864, 22, 1000), Test (288, 22, 1000)
   Subject 9: Train (864, 22, 1000), Test (288, 22, 1000)

🎯 TARGET SUBJECT: 1
   ⚙️ Applying Euclidean Alignment...
   ... Applying Euclidean Alignment (Shape: (6912, 22, 1000))
   ... Applying Euclidean Alignment (Shape: (864, 22, 1000))
   ... Applying Euclidean Alignment (Shape: (288, 22, 1000))
   ⚗️ Applying Standard Scaling...
   🏗️  Initializing EEG_Swin model...
   📊 Model Parameters: 605,540
   🎯 Model Architecture: HybridSwinCNN
   🔍

/usr/local/lib/python3.12/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4317.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


   EEG-Swin Transformer Configuration:
     - Embed Dim: 128
     - Depths: [2, 2, 6, 2]
     - Num Heads: [4, 8, 16, 32]
     - Window Size: 7
     - Num Patches: 20
      ⚠️  Using hybrid version: PatchMerging.__init__() takes from 2 to 3 positional arguments but 4 were given
   🚀 Pre-training (200 epochs)...
      Ep 20: Loss 0.4304 | Acc 96.67% | Best 96.67%
      Ep 40: Loss 0.3902 | Acc 98.39% | Best 98.39%
      Ep 60: Loss 0.3834 | Acc 98.78% | Best 99.05%
      Ep 80: Loss 0.3764 | Acc 99.07% | Best 99.44%
      Ep 100: Loss 0.3722 | Acc 99.19% | Best 99.57%
      Ep 120: Loss 0.3700 | Acc 99.28% | Best 99.64%
      Ep 140: Loss 0.3621 | Acc 99.59% | Best 99.67%
      Ep 160: Loss 0.3625 | Acc 99.61% | Best 99.68%
      Ep 180: Loss 0.3651 | Acc 99.45% | Best 99.75%
      Ep 200: Loss 0.3624 | Acc 99.61% | Best 99.84%
   🔧 Fine-tuning (50 epochs)...
      Ep 5: Loss 0.9984 | Acc 67.48% | Best 67.48%
      Ep 10: Loss 0.8132 | Acc 74.88% | Best 74.88%
      Ep 15: Loss 0.7003 |

In [ ]:
# ==========================================
# COMPLETE SOTA 6 FGTCN - FIXED VERSION
# ==========================================
import os
import numpy as np
import scipy.io
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from scipy import signal, stats, linalg
from sklearn.covariance import ledoit_wolf
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import cohen_kappa_score, f1_score
import pywt
import time
import json
import warnings
warnings.filterwarnings('ignore')

# ==================== CONFIG ====================
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Device: {DEVICE}")

# Fix random seeds for reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ==================== SIMPLIFIED SOTA HYPERPARAMETERS ====================
FS = 250
N_CLASSES = 4
N_CHANNELS = 22
TIME_WINDOW = 1000
USE_TTA = True
LABEL_SMOOTHING = 0.1
BATCH_SIZE = 64
LR_PRETRAIN = 0.001
LR_FINETUNE = 0.0005
EPOCHS_PRETRAIN = 150
EPOCHS_FINETUNE = 80
PATIENCE = 15

# Simplified parameters
MIXUP_ALPHA = 0.4
NOISE_STD = 0.01
FREQ_SHIFT_RANGE = 0.05

DATA_PATH = '/content/gdrive/MyDrive/BCICIV-2a-mat'
MODEL_NAME = 'FGTCN_SOTA_SIMPLIFIED'
SAVE_PATH = f'/content/gdrive/MyDrive/BCI_ENSEMBLE/{MODEL_NAME}'
os.makedirs(SAVE_PATH, exist_ok=True)

# ==================== SIMPLIFIED PREPROCESSING ====================
class SimplifiedPreprocessor:
    """Simplified but effective preprocessing for BCI IV 2a"""

    def __init__(self, fs=250):
        self.fs = fs
        self.scaler = RobustScaler()
        self.ea_whitening = {}

    def process_data(self, X, y=None, fit=False, subject_id=None):
        """
        Simplified but effective processing pipeline
        """
        print(f"   Processing {X.shape} samples...")

        # 1. Basic preprocessing
        X_clean = self.basic_preprocessing(X)

        # 2. Simplified feature extraction
        features = self.extract_features(X_clean)

        # 3. Domain adaptation (Euclidean Alignment only)
        if fit and subject_id is not None:
            self.fit_euclidean_alignment(X_clean, subject_id)

        if subject_id in self.ea_whitening:
            features = self.apply_euclidean_alignment(features, subject_id)

        # 4. Robust scaling
        n_samples, n_features = features.shape
        if fit:
            features_scaled = self.scaler.fit_transform(features)
        else:
            features_scaled = self.scaler.transform(features)

        return features_scaled

    def basic_preprocessing(self, X):
        """Basic but effective preprocessing"""
        n_trials, n_channels, n_times = X.shape

        # 1. Notch filter (50Hz)
        X_clean = self.apply_notch_filter(X, 50)

        # 2. Bandpass filter (4-38Hz) - optimal for motor imagery
        X_clean = self.apply_bandpass_filter(X_clean, 4, 38)

        # 3. Remove DC offset
        X_clean = X_clean - np.mean(X_clean, axis=2, keepdims=True)

        # 4. Average reference
        X_clean = X_clean - np.mean(X_clean, axis=1, keepdims=True)

        return X_clean

    def extract_features(self, X):
        """Extract key features (simplified)"""
        features_list = []

        # 1. Temporal features
        temporal = self.extract_temporal_features(X)
        features_list.append(temporal)

        # 2. Spectral features (most important)
        spectral = self.extract_spectral_features(X)
        features_list.append(spectral)

        # 3. Riemannian features (effective for cross-subject)
        riemannian = self.extract_riemannian_features(X)
        features_list.append(riemannian)

        return np.concatenate(features_list, axis=1)

    def extract_temporal_features(self, X):
        """Extract temporal features"""
        n_trials, n_channels, n_times = X.shape

        # Mean and variance
        mean = np.mean(X, axis=2, keepdims=True)
        std = np.std(X, axis=2, keepdims=True)

        # Hjorth parameters
        activity = np.var(X, axis=2, keepdims=True)
        mobility = np.std(np.diff(X, axis=2), axis=2, keepdims=True) / (std + 1e-10)

        return np.concatenate([mean, std, activity, mobility], axis=1)

    def extract_spectral_features(self, X):
        """Extract spectral features"""
        n_trials, n_channels, n_times = X.shape

        # Frequency bands for motor imagery
        bands = [
            (8, 13),    # Alpha (most important)
            (13, 30),   # Beta
            (30, 45)    # Gamma
        ]

        features = []
        for low, high in bands:
            # Band power
            filtered = self.apply_bandpass_filter(X, low, high)
            power = np.var(filtered, axis=2, keepdims=True)
            features.append(power)

            # Log power (more Gaussian)
            log_power = np.log(power + 1e-10)
            features.append(log_power)

        return np.concatenate(features, axis=1)

    def extract_riemannian_features(self, X):
        """Extract Riemannian tangent space features"""
        n_trials, n_channels, n_times = X.shape

        # Compute covariance matrices
        covariances = np.zeros((n_trials, n_channels, n_channels))
        for i in range(n_trials):
            X_i = X[i].T
            cov = X_i.T @ X_i / (n_times - 1)
            # Regularization
            cov = cov + 1e-6 * np.eye(n_channels)
            covariances[i] = cov

        # Reference matrix (identity)
        ref_matrix = np.eye(n_channels)

        # Project to tangent space (simplified)
        features = []
        for i in range(n_trials):
            # Symmetric matrix logarithm
            eigvals, eigvecs = np.linalg.eigh(covariances[i])
            eigvals = np.maximum(eigvals, 1e-10)
            log_cov = eigvecs @ np.diag(np.log(eigvals)) @ eigvecs.T

            # Upper triangular part (excluding diagonal)
            idx = np.triu_indices(n_channels, k=1)
            tangent_vec = log_cov[idx]
            features.append(tangent_vec)

        return np.array(features)

    def fit_euclidean_alignment(self, X, subject_id):
        """Fit Euclidean Alignment"""
        n_trials, n_channels, n_times = X.shape

        # Compute mean covariance
        cov_sum = np.zeros((n_channels, n_channels))
        for i in range(n_trials):
            X_i = X[i].T
            cov = X_i.T @ X_i / (n_times - 1)
            cov_sum += cov

        mean_cov = cov_sum / n_trials

        # Whitening matrix
        eigvals, eigvecs = np.linalg.eigh(mean_cov)
        eigvals = np.maximum(eigvals, 1e-10)
        whitening = eigvecs @ np.diag(1/np.sqrt(eigvals)) @ eigvecs.T

        self.ea_whitening[subject_id] = whitening

    def apply_euclidean_alignment(self, features, subject_id):
        """Apply Euclidean Alignment"""
        if subject_id in self.ea_whitening:
            whitening = self.ea_whitening[subject_id]
            n_samples, n_features = features.shape

            # If features are covariance-based, apply whitening
            # For other features, apply transformation if shape matches
            if n_features == whitening.shape[0] * whitening.shape[1]:
                # Reshape and apply
                features_reshaped = features.reshape(-1, whitening.shape[0], whitening.shape[1])
                aligned = np.zeros_like(features_reshaped)
                for i in range(features_reshaped.shape[0]):
                    aligned[i] = whitening @ features_reshaped[i] @ whitening.T
                return aligned.reshape(n_samples, -1)

        return features

    def apply_notch_filter(self, X, freq):
        """Apply notch filter"""
        nyq = 0.5 * self.fs
        freq_norm = freq / nyq
        b, a = signal.iirnotch(freq_norm, 30)
        return signal.filtfilt(b, a, X, axis=2)

    def apply_bandpass_filter(self, X, low, high):
        """Apply bandpass filter"""
        nyq = 0.5 * self.fs
        low_norm = low / nyq
        high_norm = high / nyq
        b, a = signal.butter(4, [low_norm, high_norm], btype='band')
        return signal.filtfilt(b, a, X, axis=2)

# ==================== SIMPLIFIED AUGMENTATION ====================
class SimpleAugmentation:
    """Simple but effective augmentation"""

    def __init__(self, mixup_alpha=0.4, noise_std=0.01):
        self.mixup_alpha = mixup_alpha
        self.noise_std = noise_std

    def apply_augmentations(self, X, y, training=True, n_classes=N_CLASSES):
        """Apply simple augmentations"""
        if not training:
            return X, y

        X_aug, y_aug_initial = X.clone(), y.clone()

        # 1. Add Gaussian noise
        if np.random.rand() < 0.5:
            noise = torch.randn_like(X_aug) * self.noise_std
            X_aug = X_aug + noise

        # 2. Simple MixUp
        if np.random.rand() < 0.3 and self.mixup_alpha > 0:
            batch_size = X_aug.shape[0]
            indices = torch.randperm(batch_size).to(y_aug_initial.device)

            lam = np.random.beta(self.mixup_alpha, self.mixup_alpha)
            lam = max(lam, 1 - lam)

            # Convert integer labels to one-hot encoding before MixUp
            y_onehot = torch.nn.functional.one_hot(y_aug_initial, num_classes=n_classes).float().to(y_aug_initial.device)
            y_onehot_shuffled = torch.nn.functional.one_hot(y_aug_initial[indices], num_classes=n_classes).float().to(y_aug_initial.device)

            y_aug_final = lam * y_onehot + (1 - lam) * y_onehot_shuffled
        else:
            # If MixUp is not applied, return original labels (which are LongTensor)
            y_aug_final = y_aug_initial

        return X_aug, y_aug_final

# ==================== FGTCN MODEL (ORIGINAL) ====================
# Using the original FGTCN which works well
class FGTCN(nn.Module):
    def __init__(self, n_classes=4, input_ch=22, input_time=1000):
        super(FGTCN, self).__init__()

        # Frequency Attention
        self.freq_attn = nn.Conv2d(1, 1, kernel_size=(1, 15), padding=(0, 7))
        self.freq_sigmoid = nn.Sigmoid()

        # Temporal Convolution
        self.temp_conv = nn.Conv2d(1, 16, kernel_size=(1, 64), padding=(0, 32), bias=False)
        self.bn1 = nn.BatchNorm2d(16)

        # Channel Attention
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.fc1 = nn.Conv2d(16, 16 // 8, 1, bias=False)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Conv2d(16 // 8, 16, 1, bias=False)
        self.chan_sigmoid = nn.Sigmoid()

        # Depth-wise Convolutions
        self.depth_conv1 = nn.Conv2d(16, 32, kernel_size=(input_ch, 1), groups=16)
        self.depth_conv2 = nn.Conv2d(16, 32, kernel_size=(input_ch, 1), groups=16, dilation=(1, 2))
        self.bn2 = nn.BatchNorm2d(64)
        self.act = nn.ELU()
        self.pool1 = nn.AvgPool2d(kernel_size=(1, 4))
        self.dropout1 = nn.Dropout(0.5)

        # Inception Block
        self.branch1 = nn.Conv2d(64, 32, kernel_size=(1, 1))
        self.branch2 = nn.Conv2d(64, 32, kernel_size=(1, 3), padding=(0, 1))
        self.branch3 = nn.Conv2d(64, 32, kernel_size=(1, 5), padding=(0, 2))
        self.conv_concat = nn.Conv2d(96, 32, kernel_size=1)
        self.bn3 = nn.BatchNorm2d(32)
        self.act2 = nn.ELU()
        self.pool2 = nn.AvgPool2d(kernel_size=(1, 8))
        self.dropout2 = nn.Dropout(0.5)

        # Calculate flatten dimension
        with torch.no_grad():
            dummy = torch.zeros(1, 1, input_ch, input_time)
            x = self.freq_sigmoid(self.freq_attn(dummy)) * dummy
            x = self.bn1(self.temp_conv(x))

            avg_out = self.fc2(self.relu1(self.fc1(self.avg_pool(x))))
            max_out = self.fc2(self.relu1(self.fc1(self.max_pool(x))))
            x = x * self.chan_sigmoid(avg_out + max_out)

            d1 = self.depth_conv1(x)
            d2 = self.depth_conv2(x)
            if d1.shape[3] != d2.shape[3]:
                diff = d1.shape[3] - d2.shape[3]
                d2 = nn.functional.pad(d2, (diff // 2, diff - diff // 2))

            x = self.dropout1(self.pool1(self.act(self.bn2(torch.cat([d1, d2], dim=1)))))

            b1 = self.branch1(x)
            b2 = self.branch2(x)
            b3 = self.branch3(x)
            x = self.dropout2(self.pool2(self.act2(self.bn3(self.conv_concat(torch.cat([b1, b2, b3], dim=1))))))

            self.flatten_dim = x.view(1, -1).size(1)

        self.fc = nn.Linear(self.flatten_dim, n_classes)

    def forward(self, x):
        # Frequency attention
        x = self.freq_sigmoid(self.freq_attn(x)) * x

        # Temporal convolution
        x = self.bn1(self.temp_conv(x))

        # Channel attention
        avg_out = self.fc2(self.relu1(self.fc1(self.avg_pool(x))))
        max_out = self.fc2(self.relu1(self.fc1(self.max_pool(x))))
        x = x * self.chan_sigmoid(avg_out + max_out)

        # Depth-wise convolutions
        d1 = self.depth_conv1(x)
        d2 = self.depth_conv2(x)
        if d1.shape[3] != d2.shape[3]:
            diff = d1.shape[3] - d2.shape[3]
            d2 = nn.functional.pad(d2, (diff // 2, diff - diff // 2))

        x = self.dropout1(self.pool1(self.act(self.bn2(torch.cat([d1, d2], dim=1)))))

        # Inception block
        b1 = self.branch1(x)
        b2 = self.branch2(x)
        b3 = self.branch3(x)
        x = self.dropout2(self.pool2(self.act2(self.bn3(self.conv_concat(torch.cat([b1, b2, b3], dim=1))))))

        # Classification
        x = x.view(x.size(0), -1)
        return self.fc(x)

# ==================== TRAINING FUNCTIONS ====================
def train_epoch_simple(model, loader, optimizer, criterion, augmentation=None, training=True):
    """Simple training function"""
    model.train() if training else model.eval()

    total_loss = 0
    correct = 0
    total = 0

    for inputs, labels in loader:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)

        # Apply augmentation during training
        if training and augmentation is not None:
            # Pass N_CLASSES to augmentation for one-hot encoding
            inputs, labels = augmentation.apply_augmentations(inputs, labels, training, n_classes=N_CLASSES)

        optimizer.zero_grad()

        with torch.set_grad_enabled(training):
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            if training:
                loss.backward()
                optimizer.step()

        total_loss += loss.item()
        # When labels are one-hot encoded (from MixUp), convert them back to class indices for accuracy calculation
        if labels.ndim > 1 and labels.dtype == torch.float: # If labels are one-hot (from MixUp)
            _, true_labels = torch.max(labels, 1)
        else:
            true_labels = labels # Otherwise, they are integer class indices

        _, predicted = torch.max(outputs.data, 1)
        total += true_labels.size(0)
        correct += (predicted == true_labels).sum().item()

    accuracy = 100 * correct / total
    avg_loss = total_loss / len(loader)

    return avg_loss, accuracy

def evaluate_simple(model, loader):
    """Simple evaluation"""
    model.eval()

    all_probs = []
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)

            outputs = model(inputs)
            probs = torch.softmax(outputs, dim=1)
            _, preds = torch.max(outputs, 1)

            all_probs.append(probs.cpu().numpy())
            all_preds.append(preds.cpu().numpy())
            all_labels.append(labels.cpu().numpy())

    return (np.vstack(all_probs),
            np.concatenate(all_preds),
            np.concatenate(all_labels))

# ==================== DATA LOADING ====================
def load_bci_data_raw(subject_id, base_path, is_train=True):
    """Load BCI IV 2a data"""
    file_type = 'T' if is_train else 'E'
    file_name = f"A{subject_id:02d}{file_type}.mat"
    full_path = os.path.join(base_path, file_name)

    if not os.path.exists(full_path):
        file_name = f"A0{subject_id}{file_type}.mat"
        full_path = os.path.join(base_path, file_name)
        if not os.path.exists(full_path):
            return None, None

    try:
        mat = scipy.io.loadmat(full_path)
        data_struct = mat['data']
        all_X, all_y = [], []

        for i in range(data_struct.shape[1]):
            try:
                run_data = data_struct[0][i]
                X_cnt = run_data['X'][0][0]
                trial_idx = run_data['trial'][0][0].flatten()
                y_cnt = run_data['y'][0][0].flatten()

                if len(trial_idx) == 0:
                    continue

                for j, start_idx in enumerate(trial_idx):
                    label = y_cnt[j]
                    if label not in [1, 2, 3, 4]:
                        continue

                    # Time window
                    s = start_idx - 1
                    e = s + TIME_WINDOW

                    if e <= X_cnt.shape[0]:
                        raw_epoch = X_cnt[s:e, :22].T
                        all_X.append(raw_epoch)
                        all_y.append(label - 1)
            except:
                continue

        if len(all_X) == 0:
            return None, None

        return np.stack(all_X), np.array(all_y)
    except:
        return None, None

# ==================== MAIN TRAINING LOOP ====================
print(f"\n{'='*80}")
print(f"🚀 STARTING SIMPLIFIED SOTA TRAINING")
print(f"{'='*80}")

# Cache all data
data_cache = {}
print("\n📥 Caching and preprocessing all data...")
all_subjects = list(range(1, 10))

# Initialize preprocessor
preprocessor = SimplifiedPreprocessor(fs=FS)

for s in all_subjects:
    print(f"\n   Processing Subject {s}...")

    # Load raw data
    X_tr, y_tr = load_bci_data_raw(s, DATA_PATH, True)
    X_te, y_te = load_bci_data_raw(s, DATA_PATH, False)

    if X_tr is not None and X_te is not None:
        print(f"      Raw data: Train {X_tr.shape}, Test {X_te.shape}")

        try:
            # Apply preprocessing
            X_tr_processed = preprocessor.process_data(
                X_tr, y_tr, fit=True, subject_id=s
            )
            X_te_processed = preprocessor.process_data(
                X_te, fit=False, subject_id=s
            )

            # Reshape for model
            X_tr_final = X_tr_processed.reshape(-1, 1, X_tr_processed.shape[1], 1)
            X_te_final = X_te_processed.reshape(-1, 1, X_te_processed.shape[1], 1)

            # Adjust time dimension (pad or truncate)
            target_time = 1000
            if X_tr_final.shape[3] < target_time:
                pad_size = target_time - X_tr_final.shape[3]
                X_tr_final = np.pad(X_tr_final, ((0,0), (0,0), (0,0), (0,pad_size)), mode='constant')
                X_te_final = np.pad(X_te_final, ((0,0), (0,0), (0,0), (0,pad_size)), mode='constant')
            elif X_tr_final.shape[3] > target_time:
                X_tr_final = X_tr_final[:, :, :, :target_time]
                X_te_final = X_te_final[:, :, :, :target_time]

            data_cache[s] = {
                'X_tr': torch.FloatTensor(X_tr_final),
                'y_tr': torch.LongTensor(y_tr),
                'X_te': torch.FloatTensor(X_te_final),
                'y_te': torch.LongTensor(y_te)
            }

            print(f"      Processed: Train {X_tr_final.shape}, Test {X_te_final.shape}")

        except Exception as e:
            print(f"      Error processing subject {s}: {e}")
            # Fallback: use raw data with minimal processing
            X_tr_simple = X_tr[:, :, :1000]  # Ensure 1000 time points
            X_te_simple = X_te[:, :, :1000]

            # Basic normalization
            X_tr_mean = np.mean(X_tr_simple, axis=(1,2), keepdims=True)
            X_tr_std = np.std(X_tr_simple, axis=(1,2), keepdims=True)
            X_tr_norm = (X_tr_simple - X_tr_mean) / (X_tr_std + 1e-8)

            X_te_norm = (X_te_simple - X_tr_mean) / (X_tr_std + 1e-8)

            data_cache[s] = {
                'X_tr': torch.FloatTensor(X_tr_norm[:, np.newaxis, :, :]),
                'y_tr': torch.LongTensor(y_tr),
                'X_te': torch.FloatTensor(X_te_norm[:, np.newaxis, :, :]),
                'y_te': torch.LongTensor(y_te)
            }
            print(f"      Used fallback processing")

# Initialize results
results = {}
kappa_scores = {}
f1_scores = {}

# Initialize augmentation
augmentation = SimpleAugmentation(
    mixup_alpha=MIXUP_ALPHA,
    noise_std=NOISE_STD
)

for target_sub in all_subjects:
    print(f"\n{'='*60}")
    print(f"🎯 TRAINING ON SUBJECT {target_sub}")
    print(f"{'='*60}")

    start_time = time.time()

    # Prepare source data
    X_source_list, y_source_list = [], []
    for src in all_subjects:
        if src != target_sub and src in data_cache:
            X_source_list.append(data_cache[src]['X_tr'])
            y_source_list.append(data_cache[src]['y_tr'])

    if not X_source_list:
        continue

    X_source = torch.cat(X_source_list, dim=0)
    y_source = torch.cat(y_source_list, dim=0)

    # Prepare target data
    if target_sub not in data_cache:
        continue

    X_target_train = data_cache[target_sub]['X_tr']
    y_target_train = data_cache[target_sub]['y_tr']
    X_target_test = data_cache[target_sub]['X_te']
    y_target_test = data_cache[target_sub]['y_te']

    # Create data loaders
    source_dataset = TensorDataset(X_source, y_source)
    target_train_dataset = TensorDataset(X_target_train, y_target_train)
    target_test_dataset = TensorDataset(X_target_test, y_target_test)

    source_loader = DataLoader(source_dataset, batch_size=BATCH_SIZE, shuffle=True)
    target_train_loader = DataLoader(target_train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    target_test_loader = DataLoader(target_test_dataset, batch_size=BATCH_SIZE, shuffle=False)

    # Initialize model
    input_ch = X_source.shape[2]
    input_time = X_source.shape[3]

    model = FGTCN(
        n_classes=N_CLASSES,
        input_ch=input_ch,
        input_time=input_time
    ).to(DEVICE)

    # Loss function with label smoothing
    class LabelSmoothingLoss(nn.Module):
        def __init__(self, smoothing=0.1):
            super().__init__()
            self.smoothing = smoothing
            self.log_softmax = nn.LogSoftmax(dim=1)

        def forward(self, inputs, targets):
            n_classes = inputs.size(1)
            log_prob = self.log_softmax(inputs)

            if targets.dtype == torch.long: # Original integer labels
                one_hot = torch.zeros_like(log_prob).scatter(1, targets.unsqueeze(1), 1)
                # Apply label smoothing
                one_hot = one_hot * (1 - self.smoothing) + self.smoothing / n_classes
                loss = -(one_hot * log_prob).sum(dim=1).mean()
            elif targets.dtype == torch.float: # Smoothed/mixed labels (e.g., from MixUp)
                # If targets are already a float distribution (from MixUp using one-hot),
                # apply label smoothing to these mixed one-hots.
                smoothed_targets = targets * (1 - self.smoothing) + self.smoothing / n_classes
                loss = -(smoothed_targets * log_prob).sum(dim=1).mean()
            else:
                raise TypeError(f"Unsupported target dtype: {targets.dtype}")
            return loss

    criterion = LabelSmoothingLoss(smoothing=LABEL_SMOOTHING)
    optimizer = optim.Adam(model.parameters(), lr=LR_PRETRAIN)

    # ========== PRE-TRAINING ==========
    print(f"\n   🔄 Pre-training on source data...")
    best_source_acc = 0

    for epoch in range(EPOCHS_PRETRAIN):
        loss, acc = train_epoch_simple(
            model, source_loader, optimizer, criterion,
            augmentation, training=True
        )

        if acc > best_source_acc:
            best_source_acc = acc

        if (epoch + 1) % 30 == 0:
            print(f"      Epoch {epoch+1:3d}: Loss={loss:.4f}, Acc={acc:.2f}%")

    print(f"   ✅ Best source accuracy: {best_source_acc:.2f}%")

    # ========== FINE-TUNING ==========
    print(f"\n   🔄 Fine-tuning on target data...")
    optimizer_ft = optim.Adam(model.parameters(), lr=LR_FINETUNE)
    best_target_acc = 0
    best_model_state = None

    for epoch in range(EPOCHS_FINETUNE):
        loss, acc = train_epoch_simple(
            model, target_train_loader, optimizer_ft, criterion,
            augmentation, training=True
        )

        if acc > best_target_acc:
            best_target_acc = acc
            best_model_state = model.state_dict().copy()

        if (epoch + 1) % 20 == 0:
            print(f"      Epoch {epoch+1:3d}: Loss={loss:.4f}, Acc={acc:.2f}%")

    # Load best model
    if best_model_state is not None:
        model.load_state_dict(best_model_state)

    # ========== EVALUATION ==========
    print(f"\n   📊 Evaluating on test data...")
    probs, preds, labels = evaluate_simple(model, target_test_loader)

    accuracy = 100 * (preds == labels).mean()
    kappa = cohen_kappa_score(labels, preds)
    f1 = f1_score(labels, preds, average='weighted')

    results[target_sub] = accuracy
    kappa_scores[target_sub] = kappa
    f1_scores[target_sub] = f1

    # Save results
    save_file = os.path.join(SAVE_PATH, f"S{target_sub}")
    np.save(f"{save_file}_probs.npy", probs)
    np.save(f"{save_file}_labels.npy", labels)
    np.save(f"{save_file}_preds.npy", preds)

    elapsed = time.time() - start_time

    print(f"\n   ✅ SUBJECT {target_sub} COMPLETE")
    print(f"   {'─'*35}")
    print(f"   📊 Test Performance:")
    print(f"   Accuracy:  {accuracy:.2f}%")
    print(f"   Kappa:     {kappa:.3f}")
    print(f"   F1-score:  {f1:.3f}")
    print(f"   Time:      {elapsed:.1f} seconds")

# ==================== FINAL RESULTS ====================
print(f"\n{'='*80}")
print(f"📊 FINAL RESULTS")
print(f"{'='*80}")

accuracies = list(results.values())
kappas = list(kappa_scores.values())
f1s = list(f1_scores.values())

print(f"\n{'Subject':<10} | {'Accuracy':<10} | {'Kappa':<8} | {'F1-Score':<8}")
print("-" * 45)
for sub in all_subjects:
    if sub in results:
        print(f"{sub:<10} | {results[sub]:.2f}%      | {kappa_scores[sub]:.3f}   | {f1_scores[sub]:.3f}")

print("-" * 45)
print(f"{'AVERAGE':<10} | {np.mean(accuracies):.2f}%      | {np.mean(kappas):.3f}   | {np.mean(f1s):.3f}")
print(f"{'STD DEV':<10} | {np.std(accuracies):.2f}%      | {np.std(kappas):.3f}   | {np.std(f1s):.3f}")

# Save summary
summary = {
    'model': MODEL_NAME,
    'preprocessing': 'simplified_sota',
    'augmentation': 'mixup+noise',
    'architecture': 'FGTCN',
    'results': {
        str(sub): {
            'accuracy': float(results.get(sub, 0)),
            'kappa': float(kappa_scores.get(sub, 0)),
            'f1': float(f1_scores.get(sub, 0))
        }
        for sub in all_subjects if sub in results
    },
    'overall': {
        'mean_accuracy': float(np.mean(accuracies)),
        'std_accuracy': float(np.std(accuracies)),
        'mean_kappa': float(np.mean(kappas)),
        'std_kappa': float(np.std(kappas)),
        'mean_f1': float(np.mean(f1s)),
        'std_f1': float(np.std(f1s))
    },
    'expected_range': '75-85%',
    'timestamp': time.strftime("%Y-%m-%d %H:%M:%S")
}

with open(os.path.join(SAVE_PATH, 'summary.json'), 'w') as f:
    json.dump(summary, f, indent=2)

print(f"\n✅ Training completed!")
print(f"📁 Results saved to: {SAVE_PATH}/")
print(f"🎯 Expected accuracy range: 75-85%")

✅ Device: cuda

🚀 STARTING SIMPLIFIED SOTA TRAINING

📥 Caching and preprocessing all data...

   Processing Subject 1...
      Raw data: Train (288, 22, 1000), Test (288, 22, 1000)
   Processing (288, 22, 1000) samples...
      Error processing subject 1: all the input arrays must have same number of dimensions, but the array at index 0 has 3 dimension(s) and the array at index 2 has 2 dimension(s)
      Used fallback processing

   Processing Subject 2...
      Raw data: Train (288, 22, 1000), Test (288, 22, 1000)
   Processing (288, 22, 1000) samples...
      Error processing subject 2: all the input arrays must have same number of dimensions, but the array at index 0 has 3 dimension(s) and the array at index 2 has 2 dimension(s)
      Used fallback processing

   Processing Subject 3...
      Raw data: Train (288, 22, 1000), Test (288, 22, 1000)
   Processing (288, 22, 1000) samples...
      Error processing subject 3: all the input arrays must have same number of dimensions, but th

In [ ]:
# ==========================================
# CELL 7: HYBRID BiGRU-LSTM (CNN-RNN)
# ==========================================
import os
import numpy as np
import scipy.io
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from scipy.signal import butter, lfilter
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import cohen_kappa_score, f1_score
import time
import json
import copy

# ==================== CONFIGURATION ====================
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Device: {DEVICE}")

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Data Params
FS = 250
N_CLASSES = 4
N_CHANNELS = 22
TIME_WINDOW = 1000

# RNN Training Params
BATCH_SIZE = 64
LR_PRETRAIN = 0.001       # RNNs like standard Learning Rates
LR_FINETUNE = 0.0001
EPOCHS_PRETRAIN = 100
EPOCHS_FINETUNE = 40
HIDDEN_DIM = 64           # Size of RNN memory
RNN_DROPOUT = 0.5         # Dropout inside RNNs

DATA_PATH = '/content/gdrive/MyDrive/BCICIV-2a-mat'
MODEL_NAME = 'Hybrid_BiGRU_LSTM'
SAVE_PATH = f'/content/gdrive/MyDrive/BCI_ENSEMBLE/{MODEL_NAME}'

if not os.path.exists(SAVE_PATH):
    os.makedirs(SAVE_PATH)

# ==================== UTILS ====================
def butter_bandpass_filter(data, lowcut=4.0, highcut=38.0, fs=250, order=5):
    nyq = 0.5 * fs
    b, a = butter(order, [lowcut/nyq, highcut/nyq], btype='band')
    return lfilter(b, a, data, axis=-1)

def euclidean_alignment(X_data):
    print(f"   ... Applying Euclidean Alignment (Shape: {X_data.shape})")
    covariances = np.matmul(X_data, np.transpose(X_data, (0, 2, 1)))
    mean_cov = np.mean(covariances, axis=0)
    d, v = np.linalg.eigh(mean_cov)
    d_inv_sqrt = np.diag(1.0 / np.sqrt(d + 1e-7))
    whitening_mat = np.dot(v, np.dot(d_inv_sqrt, v.T))
    X_transposed = np.transpose(X_data, (0, 2, 1))
    X_aligned = np.matmul(X_transposed, whitening_mat)
    return np.transpose(X_aligned, (0, 2, 1))

def scale_data(X, scaler, fit=False):
    n, c, t = X.shape
    x_flat = X.transpose(0, 2, 1).reshape(-1, c)
    if fit:
        x_scaled = scaler.fit_transform(x_flat)
    else:
        x_scaled = scaler.transform(x_flat)
    return x_scaled.reshape(n, t, c).transpose(0, 2, 1)

def load_bci_data_raw(subject_id, base_path, is_train=True):
    file_type = 'T' if is_train else 'E'
    file_name = f"A{subject_id:02d}{file_type}.mat"
    full_path = os.path.join(base_path, file_name)

    if not os.path.exists(full_path):
        file_name = f"A0{subject_id}{file_type}.mat"
        full_path = os.path.join(base_path, file_name)
        if not os.path.exists(full_path): return None, None

    try:
        mat = scipy.io.loadmat(full_path)
        data_struct = mat['data']
        all_X, all_y = [], []
        for i in range(data_struct.shape[1]):
            try:
                run_data = data_struct[0][i]
                X_cnt = run_data['X'][0][0]
                trial_idx = run_data['trial'][0][0].flatten()
                y_cnt = run_data['y'][0][0].flatten()
                if len(trial_idx) == 0: continue
                for j, start_idx in enumerate(trial_idx):
                    label = y_cnt[j]
                    if label not in [1, 2, 3, 4]: continue
                    # TTA
                    offsets = [0, int(0.25*FS), int(0.5*FS)] if is_train else [0]
                    for off in offsets:
                        s = (start_idx - 1) + off
                        e = s + TIME_WINDOW
                        if e <= X_cnt.shape[0]:
                            raw_epoch = X_cnt[s:e, :22].T
                            filtered = butter_bandpass_filter(raw_epoch, 4.0, 38.0, FS, 4)
                            all_X.append(filtered)
                            all_y.append(label - 1)
            except: continue
        if len(all_X) == 0: return None, None
        return np.stack(all_X), np.array(all_y)
    except: return None, None

def get_dataloader(X, y, batch_size, shuffle=True):
    tensor_x = torch.Tensor(X).unsqueeze(1)
    tensor_y = torch.LongTensor(y)
    return DataLoader(TensorDataset(tensor_x, tensor_y), batch_size=batch_size, shuffle=shuffle)

def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for inputs, labels in loader:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        # Clip grads for RNN stability
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    return total_loss / len(loader), 100 * correct / total

def evaluate_probs(model, loader):
    model.eval()
    all_probs, all_preds, all_labels = [], [], []
    with torch.no_grad():
        for inputs, labels in loader:
            inputs = inputs.to(DEVICE)
            outputs = model(inputs)
            probs = torch.softmax(outputs, dim=1)
            _, preds = torch.max(outputs, 1)
            all_probs.append(probs.cpu().numpy())
            all_preds.append(preds.cpu().numpy())
            all_labels.append(labels.numpy())
    return (np.vstack(all_probs), np.concatenate(all_preds), np.concatenate(all_labels))

# ==================== HYBRID BiGRU-LSTM MODEL ====================
class HybridRNN(nn.Module):
    def __init__(self, n_classes=4, n_channels=22, n_time=1000):
        super(HybridRNN, self).__init__()

        # 1. Feature Extractor (CNN Stem)
        # We need this to compress 1000 time points into manageable features
        self.conv_stem = nn.Sequential(
            nn.Conv2d(1, 32, (1, 25), padding=(0, 12)), # Temporal
            nn.BatchNorm2d(32),
            nn.ELU(),
            nn.AvgPool2d((1, 4)),

            nn.Conv2d(32, 64, (n_channels, 1)), # Spatial
            nn.BatchNorm2d(64),
            nn.ELU(),
            nn.AvgPool2d((1, 4)),
            nn.Dropout(0.5)
        )

        # Calculate RNN input size automatically
        # Input (1, 22, 1000) -> Conv1 -> (32, 22, 250) -> Conv2 -> (64, 1, 62)
        # The RNN will see 62 time steps, each with 64 features.
        self.rnn_input_size = 64
        self.hidden_dim = HIDDEN_DIM

        # 2. BiGRU Layer (Fast Context)
        self.bigru = nn.GRU(
            input_size=self.rnn_input_size,
            hidden_size=self.hidden_dim,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )

        # 3. LSTM Layer (Long-term Memory)
        # Input to LSTM is Hidden*2 because BiGRU is bidirectional
        self.lstm = nn.LSTM(
            input_size=self.hidden_dim * 2,
            hidden_size=self.hidden_dim,
            num_layers=1,
            batch_first=True,
            bidirectional=False
        )

        # 4. Classifier
        self.fc = nn.Sequential(
            nn.Linear(self.hidden_dim, 32),
            nn.ELU(),
            nn.Dropout(0.5),
            nn.Linear(32, n_classes)
        )

    def forward(self, x):
        # x: (B, 1, 22, 1000)

        # --- CNN Stem ---
        x = self.conv_stem(x) # (B, 64, 1, 62)
        x = x.squeeze(2)      # (B, 64, 62)
        x = x.permute(0, 2, 1) # (B, 62, 64) -> (Batch, Seq, Features) for RNN

        # --- RNN Body ---
        # BiGRU
        x, _ = self.bigru(x) # x: (B, 62, 128)

        # LSTM
        _, (hn, cn) = self.lstm(x) # hn: (1, B, 64) - We want the last hidden state

        # Take the last hidden state from the last layer
        out = hn[-1] # (B, 64)

        # --- Classification ---
        out = self.fc(out)
        return out

# ==================== MAIN LOOP ====================
print(f"\n{'='*60}")
print(f"STARTING {MODEL_NAME} - CNN + BiGRU + LSTM")
print(f"{'='*60}")

# Cache Data
data_cache = {}
print("📥 Caching Raw Data...")
all_subjects = list(range(1, 10))
for s in all_subjects:
    X_tr, y_tr = load_bci_data_raw(s, DATA_PATH, True)
    X_te, y_te = load_bci_data_raw(s, DATA_PATH, False)
    if X_tr is not None:
        data_cache[s] = {'X_tr': X_tr, 'y_tr': y_tr, 'X_te': X_te, 'y_te': y_te}

results = {}
kappa_scores = {}

for target_sub in all_subjects:
    start_time = time.time()
    print(f"\n🎯 TARGET SUBJECT: {target_sub}")

    # Prepare Data
    X_source_list, y_source_list = [], []
    for src in all_subjects:
        if src != target_sub and src in data_cache:
            X_source_list.append(data_cache[src]['X_tr'])
            y_source_list.append(data_cache[src]['y_tr'])

    if not X_source_list: continue
    X_source = np.concatenate(X_source_list)
    y_source = np.concatenate(y_source_list)

    X_tgt_tr = data_cache[target_sub]['X_tr']
    y_tgt_tr = data_cache[target_sub]['y_tr']
    X_tgt_te = data_cache[target_sub]['X_te']
    y_tgt_te = data_cache[target_sub]['y_te']

    print("   ⚙️ Euclidean Alignment...")
    X_source = euclidean_alignment(X_source)
    X_tgt_tr = euclidean_alignment(X_tgt_tr)
    X_tgt_te = euclidean_alignment(X_tgt_te)

    scaler = StandardScaler()
    X_source = scale_data(X_source, scaler, fit=True)
    X_tgt_tr = scale_data(X_tgt_tr, scaler, fit=False)
    X_tgt_te = scale_data(X_tgt_te, scaler, fit=False)

    train_loader = get_dataloader(X_source, y_source, BATCH_SIZE)
    ft_loader = get_dataloader(X_tgt_tr, y_tgt_tr, 32)
    test_loader = get_dataloader(X_tgt_te, y_tgt_te, 32, shuffle=False)

    # --- TRAIN ---
    print(f"   🚀 Pre-training...")
    model = HybridRNN(n_classes=N_CLASSES).to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=LR_PRETRAIN, weight_decay=1e-4)

    for ep in range(EPOCHS_PRETRAIN):
        loss, acc = train_epoch(model, train_loader, optimizer, criterion)
        if (ep+1) % 50 == 0:
            print(f"     Ep {ep+1}: Loss {loss:.4f} | Acc {acc:.2f}%")

    print(f"   🔧 Fine-tuning...")
    optimizer_ft = optim.AdamW(model.parameters(), lr=LR_FINETUNE, weight_decay=1e-4)

    best_acc = 0.0
    best_state = copy.deepcopy(model.state_dict())

    for ep in range(EPOCHS_FINETUNE):
        loss, acc = train_epoch(model, ft_loader, optimizer_ft, criterion)
        if acc > best_acc:
            best_acc = acc
            best_state = copy.deepcopy(model.state_dict())
        if (ep+1) % 10 == 0:
            print(f"     FT Ep {ep+1}: Loss {loss:.4f} | Acc {acc:.2f}%")

    model.load_state_dict(best_state)

    # --- EVAL ---
    probs, preds, labels = evaluate_probs(model, test_loader)
    acc = 100 * (preds == labels).mean()
    kappa = cohen_kappa_score(labels, preds)

    results[target_sub] = acc
    kappa_scores[target_sub] = kappa
    print(f"   📊 S{target_sub}: Acc {acc:.2f}% | Kappa {kappa:.3f}")

    # Save
    s_path = os.path.join(SAVE_PATH, f"S{target_sub}")
    np.save(f"{s_path}_probs.npy", probs)
    np.save(f"{s_path}_labels.npy", labels)
    np.save(f"{s_path}_preds.npy", preds)
    np.save(f"{s_path}_metrics.npy", np.array([acc, kappa]))

# ==================== SUMMARY ====================
print(f"\n{'='*60}")
print(f"FINAL RESULTS: {MODEL_NAME}")
if results:
    accs = list(results.values())
    print(f"AVG: {np.mean(accs):.2f}% | STD: {np.std(accs):.2f}")
    with open(os.path.join(SAVE_PATH, 'summary.json'), 'w') as f:
        json.dump({'acc': results, 'kappa': kappa_scores, 'mean': np.mean(accs)}, f)
    print(f"✅ Saved to {SAVE_PATH}/")

✅ Device: cuda

STARTING Hybrid_BiGRU_LSTM - CNN + BiGRU + LSTM
📥 Caching Raw Data...

🎯 TARGET SUBJECT: 1
   ⚙️ Euclidean Alignment...
   ... Applying Euclidean Alignment (Shape: (6912, 22, 1000))
   ... Applying Euclidean Alignment (Shape: (864, 22, 1000))
   ... Applying Euclidean Alignment (Shape: (288, 22, 1000))
   🚀 Pre-training...
     Ep 50: Loss 0.2300 | Acc 91.98%
     Ep 100: Loss 0.0911 | Acc 97.37%
   🔧 Fine-tuning...
     FT Ep 10: Loss 0.8927 | Acc 73.61%
     FT Ep 20: Loss 0.4478 | Acc 82.87%
     FT Ep 30: Loss 0.2932 | Acc 89.47%
     FT Ep 40: Loss 0.1586 | Acc 94.33%
   📊 S1: Acc 69.44% | Kappa 0.593

🎯 TARGET SUBJECT: 2
   ⚙️ Euclidean Alignment...
   ... Applying Euclidean Alignment (Shape: (6912, 22, 1000))
   ... Applying Euclidean Alignment (Shape: (864, 22, 1000))
   ... Applying Euclidean Alignment (Shape: (288, 22, 1000))
   🚀 Pre-training...
     Ep 50: Loss 0.2181 | Acc 92.01%
     Ep 100: Loss 0.0935 | Acc 97.18%
   🔧 Fine-tuning...
     FT Ep 10: Loss 

In [ ]:
# ==========================================
# CELL 8: LMDA-Net (LIGHTWEIGHT DUAL-ATTENTION SOTA)
# ==========================================
import os
import numpy as np
import scipy.io
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from scipy.signal import butter, lfilter
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import cohen_kappa_score, f1_score
import time
import json
import copy

# ==================== CONFIG ====================
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Device: {DEVICE}")

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

FS = 250
N_CLASSES = 4
N_CHANNELS = 22
TIME_WINDOW = 1000

# LMDA-Net Hyperparameters (Tuned for Stability)
BATCH_SIZE = 64
LR_PRETRAIN = 0.003       # Slightly higher LR for lightweight models
LR_FINETUNE = 0.0005
WEIGHT_DECAY = 0.01
EPOCHS_PRETRAIN = 120
EPOCHS_FINETUNE = 60
MIXUP_ALPHA = 0.5         # Crucial for small datasets
LABEL_SMOOTHING = 0.1

DATA_PATH = '/content/gdrive/MyDrive/BCICIV-2a-mat'
MODEL_NAME = 'LMDA_Net_SOTA'
SAVE_PATH = f'/content/gdrive/MyDrive/BCI_ENSEMBLE/{MODEL_NAME}'

if not os.path.exists(SAVE_PATH):
    os.makedirs(SAVE_PATH)

# ==================== UTILS ====================
def butter_bandpass_filter(data, lowcut=4.0, highcut=38.0, fs=250, order=5):
    nyq = 0.5 * fs
    b, a = butter(order, [lowcut/nyq, highcut/nyq], btype='band')
    return lfilter(b, a, data, axis=-1)

def euclidean_alignment(X_data):
    print(f"   ... Applying Euclidean Alignment (Shape: {X_data.shape})")
    covariances = np.matmul(X_data, np.transpose(X_data, (0, 2, 1)))
    mean_cov = np.mean(covariances, axis=0)
    d, v = np.linalg.eigh(mean_cov)
    d_inv_sqrt = np.diag(1.0 / np.sqrt(d + 1e-7))
    whitening_mat = np.dot(v, np.dot(d_inv_sqrt, v.T))
    X_transposed = np.transpose(X_data, (0, 2, 1))
    X_aligned = np.matmul(X_transposed, whitening_mat)
    return np.transpose(X_aligned, (0, 2, 1))

def scale_data(X, scaler, fit=False):
    n, c, t = X.shape
    x_flat = X.transpose(0, 2, 1).reshape(-1, c)
    if fit:
        x_scaled = scaler.fit_transform(x_flat)
    else:
        x_scaled = scaler.transform(x_flat)
    return x_scaled.reshape(n, t, c).transpose(0, 2, 1)

def load_bci_data_raw(subject_id, base_path, is_train=True):
    file_type = 'T' if is_train else 'E'
    file_name = f"A{subject_id:02d}{file_type}.mat"
    full_path = os.path.join(base_path, file_name)
    if not os.path.exists(full_path):
        file_name = f"A0{subject_id}{file_type}.mat"
        full_path = os.path.join(base_path, file_name)
        if not os.path.exists(full_path): return None, None
    try:
        mat = scipy.io.loadmat(full_path)
        data_struct = mat['data']
        all_X, all_y = [], []
        for i in range(data_struct.shape[1]):
            try:
                run_data = data_struct[0][i]
                X_cnt = run_data['X'][0][0]
                trial_idx = run_data['trial'][0][0].flatten()
                y_cnt = run_data['y'][0][0].flatten()
                if len(trial_idx) == 0: continue
                for j, start_idx in enumerate(trial_idx):
                    label = y_cnt[j]
                    if label not in [1, 2, 3, 4]: continue
                    # Sliding Window Augmentation
                    offsets = [0, int(0.25*FS), int(0.5*FS)] if is_train else [0]
                    for off in offsets:
                        s = (start_idx - 1) + off
                        e = s + TIME_WINDOW
                        if e <= X_cnt.shape[0]:
                            raw_epoch = X_cnt[s:e, :22].T
                            filtered = butter_bandpass_filter(raw_epoch, 4.0, 38.0, FS, 4)
                            all_X.append(filtered)
                            all_y.append(label - 1)
            except: continue
        if len(all_X) == 0: return None, None
        return np.stack(all_X), np.array(all_y)
    except: return None, None

def get_dataloader(X, y, batch_size, shuffle=True):
    tensor_x = torch.Tensor(X).unsqueeze(1) # (B, 1, 22, 1000)
    tensor_y = torch.LongTensor(y)
    return DataLoader(TensorDataset(tensor_x, tensor_y), batch_size=batch_size, shuffle=shuffle)

# ==================== MIXUP UTILS ====================
def mixup_data(x, y, alpha=0.5):
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1
    batch_size = x.size(0)
    index = torch.randperm(batch_size).to(x.device)
    mixed_x = lam * x + (1 - lam) * x[index, :]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for inputs, labels in loader:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)

        # MixUp
        inputs, targets_a, targets_b, lam = mixup_data(inputs, labels, MIXUP_ALPHA)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = mixup_criterion(criterion, outputs, targets_a, targets_b, lam)
        loss.backward()

        # Gradient Clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        optimizer.step()

        total_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (lam * predicted.eq(targets_a.data).cpu().sum().float()
                    + (1 - lam) * predicted.eq(targets_b.data).cpu().sum().float())
    return total_loss / len(loader), 100 * correct / total

def evaluate_probs(model, loader):
    model.eval()
    all_probs, all_preds, all_labels = [], [], []
    with torch.no_grad():
        for inputs, labels in loader:
            inputs = inputs.to(DEVICE)
            outputs = model(inputs)
            probs = torch.softmax(outputs, dim=1)
            _, preds = torch.max(outputs, 1)
            all_probs.append(probs.cpu().numpy())
            all_preds.append(preds.cpu().numpy())
            all_labels.append(labels.numpy())
    return (np.vstack(all_probs), np.concatenate(all_preds), np.concatenate(all_labels))

# ==================== LMDA-NET ARCHITECTURE ====================
class ChannelAttention(nn.Module):
    """ Squeeze-and-Excitation style Channel Attention """
    def __init__(self, in_planes, ratio=8):
        super(ChannelAttention, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)

        self.fc1 = nn.Conv2d(in_planes, in_planes // ratio, 1, bias=False)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Conv2d(in_planes // ratio, in_planes, 1, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = self.fc2(self.relu1(self.fc1(self.avg_pool(x))))
        max_out = self.fc2(self.relu1(self.fc1(self.max_pool(x))))
        out = avg_out + max_out
        return self.sigmoid(out)

class DepthAttention(nn.Module):
    """ Attention along the Temporal Dimension (Depth) """
    def __init__(self, in_planes, kernel_size=7):
        super(DepthAttention, self).__init__()
        padding = 3 if kernel_size == 7 else 1
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=padding, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # Average & Max across channels
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        x_cat = torch.cat([avg_out, max_out], dim=1)
        out = self.conv(x_cat)
        return self.sigmoid(out)

class LMDA_Net(nn.Module):
    def __init__(self, n_classes=4, n_channels=22, n_time=1000):
        super(LMDA_Net, self).__init__()

        # 1. Temporal Conv
        self.conv1 = nn.Conv2d(1, 16, (1, 64), (1, 2), (0, 32), bias=False)
        self.bn1 = nn.BatchNorm2d(16)

        # 2. Depthwise Spatial Conv
        self.conv2 = nn.Conv2d(16, 32, (n_channels, 1), groups=16, bias=False)
        self.bn2 = nn.BatchNorm2d(32)
        self.act1 = nn.Hardswish()
        self.pool1 = nn.AvgPool2d((1, 4))
        self.dropout1 = nn.Dropout(0.5)

        # 3. LMDA Block 1 (Dual Attention)
        self.conv3 = nn.Conv2d(32, 32, (1, 16), groups=32, padding=(0, 8), bias=False) # Separable
        self.conv4 = nn.Conv2d(32, 64, (1, 1), bias=False) # Pointwise
        self.bn3 = nn.BatchNorm2d(64)
        self.act2 = nn.Hardswish()
        self.pool2 = nn.AvgPool2d((1, 4))
        self.dropout2 = nn.Dropout(0.5)

        # Attention Modules
        self.ca = ChannelAttention(64)
        self.da = DepthAttention(64)

        # Classification
        # Calculate flatten size: 1000 -> /2 (conv1) -> /4 (pool1) -> /4 (pool2) = ~31
        self.flatten_dim = 64 * 31

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(self.flatten_dim, n_classes)
        )

    def forward(self, x):
        # x: (B, 1, 22, 1000)

        # Stem
        x = self.conv1(x)
        x = self.bn1(x)

        x = self.conv2(x)
        x = self.bn2(x)
        x = self.act1(x)
        x = self.pool1(x)
        x = self.dropout1(x)

        # Block 2
        x = self.conv3(x)
        x = self.conv4(x)
        x = self.bn3(x)
        x = self.act2(x)
        x = self.pool2(x)
        x = self.dropout2(x)

        # Dual Attention
        x = x * self.ca(x) # Channel Attention
        x = x * self.da(x) # Depth Attention (Temporal)

        x = self.classifier(x)
        return x

# ==================== MAIN LOOP ====================
print(f"\n{'='*60}")
print(f"STARTING {MODEL_NAME} - LIGHTWEIGHT SOTA")
print(f"{'='*60}")

# Cache Data
data_cache = {}
print("📥 Caching Raw Data...")
all_subjects = list(range(1, 10))
for s in all_subjects:
    X_tr, y_tr = load_bci_data_raw(s, DATA_PATH, True)
    X_te, y_te = load_bci_data_raw(s, DATA_PATH, False)
    if X_tr is not None:
        data_cache[s] = {'X_tr': X_tr, 'y_tr': y_tr, 'X_te': X_te, 'y_te': y_te}

results = {}
kappa_scores = {}

for target_sub in all_subjects:
    start_time = time.time()
    print(f"\n🎯 TARGET SUBJECT: {target_sub}")

    # Prepare Data
    X_source_list, y_source_list = [], []
    for src in all_subjects:
        if src != target_sub and src in data_cache:
            X_source_list.append(data_cache[src]['X_tr'])
            y_source_list.append(data_cache[src]['y_tr'])

    if not X_source_list: continue
    X_source = np.concatenate(X_source_list)
    y_source = np.concatenate(y_source_list)
    X_tgt_tr = data_cache[target_sub]['X_tr']
    y_tgt_tr = data_cache[target_sub]['y_tr']
    X_tgt_te = data_cache[target_sub]['X_te']
    y_tgt_te = data_cache[target_sub]['y_te']

    print("   ⚙️ Euclidean Alignment...")
    X_source = euclidean_alignment(X_source)
    X_tgt_tr = euclidean_alignment(X_tgt_tr)
    X_tgt_te = euclidean_alignment(X_tgt_te)

    scaler = StandardScaler()
    X_source = scale_data(X_source, scaler, fit=True)
    X_tgt_tr = scale_data(X_tgt_tr, scaler, fit=False)
    X_tgt_te = scale_data(X_tgt_te, scaler, fit=False)

    train_loader = get_dataloader(X_source, y_source, BATCH_SIZE)
    ft_loader = get_dataloader(X_tgt_tr, y_tgt_tr, 32)
    test_loader = get_dataloader(X_tgt_te, y_tgt_te, 32, shuffle=False)

    # --- 1. PRE-TRAINING ---
    print(f"   🚀 Pre-training (MixUp)...")
    model = LMDA_Net(n_classes=N_CLASSES).to(DEVICE)
    criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    optimizer_pre = optim.AdamW(model.parameters(), lr=LR_PRETRAIN, weight_decay=WEIGHT_DECAY)

    for ep in range(EPOCHS_PRETRAIN):
        loss, acc = train_epoch(model, train_loader, optimizer_pre, criterion)
        if (ep+1) % 50 == 0:
            print(f"     Ep {ep+1}: Loss {loss:.4f} | Acc {acc:.2f}%")

    # --- 2. FINE-TUNING (SAVE BEST) ---
    print(f"   🔧 Fine-tuning...")
    optimizer_ft = optim.AdamW(model.parameters(), lr=LR_FINETUNE, weight_decay=WEIGHT_DECAY)

    best_ft_acc = 0.0
    best_model_state = copy.deepcopy(model.state_dict())

    for ep in range(EPOCHS_FINETUNE):
        loss, acc = train_epoch(model, ft_loader, optimizer_ft, criterion)

        if acc > best_ft_acc:
            best_ft_acc = acc
            best_model_state = copy.deepcopy(model.state_dict())

        if (ep+1) % 10 == 0:
             print(f"     Ep {ep+1}: Loss {loss:.4f} | Acc {acc:.2f}% | Best {best_ft_acc:.2f}%")

    model.load_state_dict(best_model_state)

    # --- 3. EVALUATION ---
    try:
        probs, preds, labels = evaluate_probs(model, test_loader)
        acc = 100 * (preds == labels).mean()
        kappa = cohen_kappa_score(labels, preds)
        f1 = f1_score(labels, preds, average='weighted')

        save_file = os.path.join(SAVE_PATH, f"S{target_sub}")
        np.save(f"{save_file}_probs.npy", probs)
        np.save(f"{save_file}_labels.npy", labels)
        np.save(f"{save_file}_preds.npy", preds)
        np.save(f"{save_file}_metrics.npy", np.array([acc, kappa, f1]))

        results[target_sub] = acc
        kappa_scores[target_sub] = kappa
        print(f"   📊 Result: Acc {acc:.2f}% | Kappa {kappa:.3f}")
    except Exception as e:
        print(f"   ❌ Eval Failed: {e}")

# ==================== SUMMARY ====================
print(f"\n{'='*60}")
print(f"FINAL RESULTS: {MODEL_NAME}")
if results:
    accs = list(results.values())
    for sub in all_subjects:
        if sub in results:
            print(f"S{sub}: {results[sub]:.2f}% (K={kappa_scores[sub]:.3f})")
    print("-" * 35)
    print(f"AVG: {np.mean(accs):.2f}% | STD: {np.std(accs):.2f}")

    with open(os.path.join(SAVE_PATH, 'summary.json'), 'w') as f:
        json.dump({'acc': results, 'kappa': kappa_scores, 'mean': np.mean(accs)}, f)
    print(f"✅ Training completed! Results saved to {SAVE_PATH}/")

✅ Device: cuda

STARTING LMDA_Net_SOTA - LIGHTWEIGHT SOTA
📥 Caching Raw Data...

🎯 TARGET SUBJECT: 1
   ⚙️ Euclidean Alignment...
   ... Applying Euclidean Alignment (Shape: (6912, 22, 1000))
   ... Applying Euclidean Alignment (Shape: (864, 22, 1000))
   ... Applying Euclidean Alignment (Shape: (288, 22, 1000))
   🚀 Pre-training (MixUp)...
     Ep 50: Loss 0.9905 | Acc 67.36%
     Ep 100: Loss 0.9159 | Acc 72.75%
   🔧 Fine-tuning...
     Ep 10: Loss 0.9528 | Acc 69.47% | Best 71.58%
     Ep 20: Loss 0.8822 | Acc 73.36% | Best 74.70%
     Ep 30: Loss 0.8923 | Acc 74.07% | Best 78.69%
     Ep 40: Loss 0.8633 | Acc 76.78% | Best 83.35%
     Ep 50: Loss 0.8618 | Acc 77.34% | Best 83.47%
     Ep 60: Loss 0.7804 | Acc 82.28% | Best 86.70%
   📊 Result: Acc 74.31% | Kappa 0.657

🎯 TARGET SUBJECT: 2
   ⚙️ Euclidean Alignment...
   ... Applying Euclidean Alignment (Shape: (6912, 22, 1000))
   ... Applying Euclidean Alignment (Shape: (864, 22, 1000))
   ... Applying Euclidean Alignment (Shape: (

In [ ]:
# ==========================================
# CELL 9: TSception (TEMPORAL-SPATIAL INCEPTION SOTA)
# ==========================================
import os
import numpy as np
import scipy.io
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from scipy.signal import butter, lfilter
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import cohen_kappa_score, f1_score
import time
import json
import copy

# ==================== CONFIGURATION ====================
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Device: {DEVICE}")

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

FS = 250
N_CLASSES = 4
N_CHANNELS = 22
TIME_WINDOW = 1000

# TSception Tuning
BATCH_SIZE = 64
LR_PRETRAIN = 0.001
LR_FINETUNE = 0.0001
WEIGHT_DECAY = 0.01
EPOCHS_PRETRAIN = 120
EPOCHS_FINETUNE = 60
LABEL_SMOOTHING = 0.2      # High smoothing for Inception models

DATA_PATH = '/content/gdrive/MyDrive/BCICIV-2a-mat'
MODEL_NAME = 'TSception_SOTA'
SAVE_PATH = f'/content/gdrive/MyDrive/BCI_ENSEMBLE/{MODEL_NAME}'

if not os.path.exists(SAVE_PATH):
    os.makedirs(SAVE_PATH)

# ==================== UTILS ====================
def butter_bandpass_filter(data, lowcut=4.0, highcut=38.0, fs=250, order=5):
    nyq = 0.5 * fs
    b, a = butter(order, [lowcut/nyq, highcut/nyq], btype='band')
    return lfilter(b, a, data, axis=-1)

def euclidean_alignment(X_data):
    # print(f"   ... Applying Euclidean Alignment (Shape: {X_data.shape})")
    covariances = np.matmul(X_data, np.transpose(X_data, (0, 2, 1)))
    mean_cov = np.mean(covariances, axis=0)
    d, v = np.linalg.eigh(mean_cov)
    d_inv_sqrt = np.diag(1.0 / np.sqrt(d + 1e-7))
    whitening_mat = np.dot(v, np.dot(d_inv_sqrt, v.T))
    X_transposed = np.transpose(X_data, (0, 2, 1))
    X_aligned = np.matmul(X_transposed, whitening_mat)
    return np.transpose(X_aligned, (0, 2, 1))

def scale_data(X, scaler, fit=False):
    n, c, t = X.shape
    x_flat = X.transpose(0, 2, 1).reshape(-1, c)
    if fit:
        x_scaled = scaler.fit_transform(x_flat)
    else:
        x_scaled = scaler.transform(x_flat)
    return x_scaled.reshape(n, t, c).transpose(0, 2, 1)

def load_bci_data_raw(subject_id, base_path, is_train=True):
    file_type = 'T' if is_train else 'E'
    file_name = f"A{subject_id:02d}{file_type}.mat"
    full_path = os.path.join(base_path, file_name)
    if not os.path.exists(full_path):
        file_name = f"A0{subject_id}{file_type}.mat"
        full_path = os.path.join(base_path, file_name)
        if not os.path.exists(full_path): return None, None
    try:
        mat = scipy.io.loadmat(full_path)
        data_struct = mat['data']
        all_X, all_y = [], []
        for i in range(data_struct.shape[1]):
            try:
                run_data = data_struct[0][i]
                X_cnt = run_data['X'][0][0]
                trial_idx = run_data['trial'][0][0].flatten()
                y_cnt = run_data['y'][0][0].flatten()
                if len(trial_idx) == 0: continue
                for j, start_idx in enumerate(trial_idx):
                    label = y_cnt[j]
                    if label not in [1, 2, 3, 4]: continue
                    # Sliding Window
                    offsets = [0, int(0.25*FS), int(0.5*FS)] if is_train else [0]
                    for off in offsets:
                        s = (start_idx - 1) + off
                        e = s + TIME_WINDOW
                        if e <= X_cnt.shape[0]:
                            raw_epoch = X_cnt[s:e, :22].T
                            filtered = butter_bandpass_filter(raw_epoch, 4.0, 38.0, FS, 4)
                            all_X.append(filtered)
                            all_y.append(label - 1)
            except: continue
        if len(all_X) == 0: return None, None
        return np.stack(all_X), np.array(all_y)
    except: return None, None

def get_dataloader(X, y, batch_size, shuffle=True):
    tensor_x = torch.Tensor(X).unsqueeze(1) # (B, 1, 22, 1000)
    tensor_y = torch.LongTensor(y)
    return DataLoader(TensorDataset(tensor_x, tensor_y), batch_size=batch_size, shuffle=shuffle)

def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for inputs, labels in loader:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    return total_loss / len(loader), 100 * correct / total

def evaluate_probs(model, loader):
    model.eval()
    all_probs, all_preds, all_labels = [], [], []
    with torch.no_grad():
        for inputs, labels in loader:
            inputs = inputs.to(DEVICE)
            outputs = model(inputs)
            probs = torch.softmax(outputs, dim=1)
            _, preds = torch.max(outputs, 1)
            all_probs.append(probs.cpu().numpy())
            all_preds.append(preds.cpu().numpy())
            all_labels.append(labels.numpy())
    return (np.vstack(all_probs), np.concatenate(all_preds), np.concatenate(all_labels))

# ==================== TSCEPTION ARCHITECTURE ====================
class TSception(nn.Module):
    def __init__(self, n_classes=4, n_channels=22, n_time=1000, sampling_rate=250):
        super(TSception, self).__init__()
        # 1. Temporal Inception
        self.kernel_1 = int(sampling_rate * 0.5)
        self.kernel_2 = int(sampling_rate * 0.25)
        self.kernel_3 = int(sampling_rate * 0.125)

        self.t_conv1 = nn.Conv2d(1, 9, (1, self.kernel_1), padding=(0, self.kernel_1//2), bias=False)
        self.t_conv2 = nn.Conv2d(1, 9, (1, self.kernel_2), padding=(0, self.kernel_2//2), bias=False)
        self.t_conv3 = nn.Conv2d(1, 9, (1, self.kernel_3), padding=(0, self.kernel_3//2), bias=False)
        self.bn_t = nn.BatchNorm2d(27)

        # 2. Spatial Inception
        self.s_conv1 = nn.Conv2d(27, 27, (n_channels, 1), bias=False)
        self.s_conv2 = nn.Conv2d(27, 27, (n_channels, 1), bias=False)
        self.bn_s = nn.BatchNorm2d(54)

        # 3. Fusion
        self.act = nn.Mish()
        self.pool = nn.AvgPool2d((1, 8))
        self.dropout = nn.Dropout(0.5)
        self.flatten_dim = 54 * 125
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(self.flatten_dim, 128),
            nn.Mish(),
            nn.Dropout(0.5),
            nn.Linear(128, n_classes)
        )

    def forward(self, x):
        t1 = self.t_conv1(x)
        t2 = self.t_conv2(x)
        t3 = self.t_conv3(x)

        # Pad fix
        if t1.shape[3] != x.shape[3]: t1 = t1[:, :, :, :x.shape[3]]
        if t2.shape[3] != x.shape[3]: t2 = t2[:, :, :, :x.shape[3]]
        if t3.shape[3] != x.shape[3]: t3 = t3[:, :, :, :x.shape[3]]

        x = torch.cat([t1, t2, t3], dim=1)
        x = self.bn_t(x)
        x = self.act(x)

        s1 = self.s_conv1(x)
        s2 = self.s_conv2(x)
        x = torch.cat([s1, s2], dim=1)
        x = self.bn_s(x)
        x = self.act(x)

        x = self.pool(x)
        x = self.dropout(x)
        x = self.classifier(x)
        return x

# ==================== MAIN LOOP ====================
print(f"\n{'='*60}")
print(f"STARTING {MODEL_NAME} - MULTI-SCALE INCEPTION")
print(f"{'='*60}")

# Cache Data
data_cache = {}
print("📥 Caching Raw Data (Required for Source Transfer)...")
all_subjects = list(range(1, 10))
for s in all_subjects:
    X_tr, y_tr = load_bci_data_raw(s, DATA_PATH, True)
    X_te, y_te = load_bci_data_raw(s, DATA_PATH, False)
    if X_tr is not None:
        data_cache[s] = {'X_tr': X_tr, 'y_tr': y_tr, 'X_te': X_te, 'y_te': y_te}

results = {}
kappa_scores = {}

for target_sub in all_subjects:
    print(f"\n🎯 TARGET SUBJECT: {target_sub}")

    # --- CHECK IF ALREADY TRAINED ---
    save_file_base = os.path.join(SAVE_PATH, f"S{target_sub}")
    metrics_path = f"{save_file_base}_metrics.npy"

    if os.path.exists(metrics_path):
        print(f"   ⏩ Found saved metrics for S{target_sub}. Loading...")
        try:
            metrics = np.load(metrics_path)
            # metrics saved as [acc, kappa, f1]
            acc, kappa = metrics[0], metrics[1]
            results[target_sub] = acc
            kappa_scores[target_sub] = kappa
            print(f"   📊 Loaded Result: Acc {acc:.2f}% | Kappa {kappa:.3f}")
            continue # Skip to next subject
        except Exception as e:
            print(f"   ⚠️ Error loading saved file ({e}). Retraining...")

    # --- PREPARE DATA IF NOT SKIPPED ---
    start_time = time.time()

    X_source_list, y_source_list = [], []
    for src in all_subjects:
        if src != target_sub and src in data_cache:
            X_source_list.append(data_cache[src]['X_tr'])
            y_source_list.append(data_cache[src]['y_tr'])

    if not X_source_list:
        print("   ❌ No source data available.")
        continue

    X_source = np.concatenate(X_source_list)
    y_source = np.concatenate(y_source_list)
    X_tgt_tr = data_cache[target_sub]['X_tr']
    y_tgt_tr = data_cache[target_sub]['y_tr']
    X_tgt_te = data_cache[target_sub]['X_te']
    y_tgt_te = data_cache[target_sub]['y_te']

    print("   ⚙️ Euclidean Alignment...")
    X_source = euclidean_alignment(X_source)
    X_tgt_tr = euclidean_alignment(X_tgt_tr)
    X_tgt_te = euclidean_alignment(X_tgt_te)

    scaler = StandardScaler()
    X_source = scale_data(X_source, scaler, fit=True)
    X_tgt_tr = scale_data(X_tgt_tr, scaler, fit=False)
    X_tgt_te = scale_data(X_tgt_te, scaler, fit=False)

    train_loader = get_dataloader(X_source, y_source, BATCH_SIZE)
    ft_loader = get_dataloader(X_tgt_tr, y_tgt_tr, 32)
    test_loader = get_dataloader(X_tgt_te, y_tgt_te, 32, shuffle=False)

    # --- 1. PRE-TRAINING ---
    print(f"   🚀 Pre-training...")
    model = TSception(n_classes=N_CLASSES).to(DEVICE)
    criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    optimizer_pre = optim.AdamW(model.parameters(), lr=LR_PRETRAIN, weight_decay=WEIGHT_DECAY)

    for ep in range(EPOCHS_PRETRAIN):
        loss, acc = train_epoch(model, train_loader, optimizer_pre, criterion)
        if (ep+1) % 50 == 0:
            print(f"     Ep {ep+1}: Loss {loss:.4f} | Acc {acc:.2f}%")

    # --- 2. FINE-TUNING (SAVE BEST) ---
    print(f"   🔧 Fine-tuning...")
    optimizer_ft = optim.AdamW(model.parameters(), lr=LR_FINETUNE, weight_decay=WEIGHT_DECAY)

    best_ft_acc = 0.0
    best_model_state = copy.deepcopy(model.state_dict())

    for ep in range(EPOCHS_FINETUNE):
        loss, acc = train_epoch(model, ft_loader, optimizer_ft, criterion)

        if acc > best_ft_acc:
            best_ft_acc = acc
            best_model_state = copy.deepcopy(model.state_dict())

        if (ep+1) % 10 == 0:
             print(f"     Ep {ep+1}: Loss {loss:.4f} | Acc {acc:.2f}% | Best {best_ft_acc:.2f}%")

    model.load_state_dict(best_model_state)

    # --- 3. EVALUATION ---
    try:
        probs, preds, labels = evaluate_probs(model, test_loader)
        acc = 100 * (preds == labels).mean()
        kappa = cohen_kappa_score(labels, preds)
        f1 = f1_score(labels, preds, average='weighted')

        # Save Logic
        np.save(f"{save_file_base}_probs.npy", probs)
        np.save(f"{save_file_base}_labels.npy", labels)
        np.save(f"{save_file_base}_preds.npy", preds)
        np.save(f"{save_file_base}_metrics.npy", np.array([acc, kappa, f1]))

        results[target_sub] = acc
        kappa_scores[target_sub] = kappa
        print(f"   📊 Result: Acc {acc:.2f}% | Kappa {kappa:.3f}")
        print(f"   ⏱️ Time: {time.time() - start_time:.1f}s")

    except Exception as e:
        print(f"   ❌ Eval Failed: {e}")

# ==================== SUMMARY & STOPPING ====================
print(f"\n{'='*60}")
print(f"FINAL RESULTS: {MODEL_NAME}")
print(f"{'='*60}")

if results:
    # Ensure we print results sorted by subject ID
    sorted_subs = sorted(results.keys())
    accs = [results[s] for s in sorted_subs]

    for sub in sorted_subs:
        print(f"S{sub}: {results[sub]:.2f}% (K={kappa_scores[sub]:.3f})")

    print("-" * 35)
    mean_acc = np.mean(accs)
    std_acc = np.std(accs)
    print(f"AVG: {mean_acc:.2f}% | STD: {std_acc:.2f}")

    # Helper to convert numpy float32 to native float for JSON serialization
    def convert_types(obj):
        if isinstance(obj, np.integer): return int(obj)
        if isinstance(obj, np.floating): return float(obj)
        if isinstance(obj, np.ndarray): return obj.tolist()
        return obj

    final_data = {
        'acc': {k: convert_types(v) for k, v in results.items()},
        'kappa': {k: convert_types(v) for k, v in kappa_scores.items()},
        'mean': convert_types(mean_acc),
        'std': convert_types(std_acc)
    }

    with open(os.path.join(SAVE_PATH, 'summary.json'), 'w') as f:
        json.dump(final_data, f, indent=4)

    print(f"✅ Training completed! Results saved to {SAVE_PATH}/")
else:
    print("❌ No results found or generated.")

✅ Device: cuda

STARTING TSception_SOTA - MULTI-SCALE INCEPTION
📥 Caching Raw Data (Required for Source Transfer)...

🎯 TARGET SUBJECT: 1
   ⏩ Found saved metrics for S1. Loading...
   📊 Loaded Result: Acc 71.18% | Kappa 0.616

🎯 TARGET SUBJECT: 2
   ⏩ Found saved metrics for S2. Loading...
   📊 Loaded Result: Acc 54.17% | Kappa 0.389

🎯 TARGET SUBJECT: 3
   ⏩ Found saved metrics for S3. Loading...
   📊 Loaded Result: Acc 74.65% | Kappa 0.662

🎯 TARGET SUBJECT: 4
   ⏩ Found saved metrics for S4. Loading...
   📊 Loaded Result: Acc 63.89% | Kappa 0.519

🎯 TARGET SUBJECT: 5
   ⏩ Found saved metrics for S5. Loading...
   📊 Loaded Result: Acc 56.94% | Kappa 0.426

🎯 TARGET SUBJECT: 6
   ⏩ Found saved metrics for S6. Loading...
   📊 Loaded Result: Acc 50.69% | Kappa 0.343

🎯 TARGET SUBJECT: 7
   ⏩ Found saved metrics for S7. Loading...
   📊 Loaded Result: Acc 71.88% | Kappa 0.625

🎯 TARGET SUBJECT: 8
   ⏩ Found saved metrics for S8. Loading...
   📊 Loaded Result: Acc 71.88% | Kappa 0.625

🎯 

In [ ]:
# ==========================================
# CELL 10: GOLD STANDARD HYBRID (CNN + TRANSFORMER)
# ==========================================
import os
import numpy as np
import scipy.io
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from scipy.signal import butter, lfilter
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import cohen_kappa_score, f1_score
import time
import json
import copy
import math

# ==================== CONFIGURATION ====================
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Device: {DEVICE}")

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True

# Data Params
FS = 250
N_CLASSES = 4
N_CHANNELS = 22
TIME_WINDOW = 1000

# Training Params (Restored to Code 1's high-performance settings)
BATCH_SIZE = 32           # Perfect balance for BCI
LR_PRETRAIN = 0.001       # Standard AdamW start
LR_FINETUNE = 0.0001      # Lower for fine-tuning
WEIGHT_DECAY = 0.01       # Standard regularization
EPOCHS_PRETRAIN = 100
EPOCHS_FINETUNE = 60      # Sufficient for convergence

DATA_PATH = '/content/gdrive/MyDrive/BCICIV-2a-mat'
MODEL_NAME = 'Gold_Standard_Hybrid'
SAVE_PATH = f'/content/gdrive/MyDrive/BCI_ENSEMBLE/{MODEL_NAME}'

if not os.path.exists(SAVE_PATH):
    os.makedirs(SAVE_PATH)

# ==================== UTILS ====================
def butter_bandpass_filter(data, lowcut=4.0, highcut=38.0, fs=250, order=5):
    nyq = 0.5 * fs
    b, a = butter(order, [lowcut/nyq, highcut/nyq], btype='band')
    return lfilter(b, a, data, axis=-1)

def euclidean_alignment(X_data):
    print(f"   ... Applying Euclidean Alignment (Shape: {X_data.shape})")
    covariances = np.matmul(X_data, np.transpose(X_data, (0, 2, 1)))
    mean_cov = np.mean(covariances, axis=0)
    d, v = np.linalg.eigh(mean_cov)
    d_inv_sqrt = np.diag(1.0 / np.sqrt(d + 1e-7))
    whitening_mat = np.dot(v, np.dot(d_inv_sqrt, v.T))
    X_transposed = np.transpose(X_data, (0, 2, 1))
    X_aligned = np.matmul(X_transposed, whitening_mat)
    return np.transpose(X_aligned, (0, 2, 1))

def scale_data(X, scaler, fit=False):
    n, c, t = X.shape
    x_flat = X.transpose(0, 2, 1).reshape(-1, c)
    if fit:
        x_scaled = scaler.fit_transform(x_flat)
    else:
        x_scaled = scaler.transform(x_flat)
    return x_scaled.reshape(n, t, c).transpose(0, 2, 1)

def load_bci_data_raw(subject_id, base_path, is_train=True):
    file_type = 'T' if is_train else 'E'
    file_name = f"A{subject_id:02d}{file_type}.mat"
    full_path = os.path.join(base_path, file_name)

    if not os.path.exists(full_path):
        file_name = f"A0{subject_id}{file_type}.mat"
        full_path = os.path.join(base_path, file_name)
        if not os.path.exists(full_path): return None, None

    try:
        mat = scipy.io.loadmat(full_path)
        data_struct = mat['data']
        all_X, all_y = [], []
        for i in range(data_struct.shape[1]):
            try:
                run_data = data_struct[0][i]
                X_cnt = run_data['X'][0][0]
                trial_idx = run_data['trial'][0][0].flatten()
                y_cnt = run_data['y'][0][0].flatten()
                if len(trial_idx) == 0: continue
                for j, start_idx in enumerate(trial_idx):
                    label = y_cnt[j]
                    if label not in [1, 2, 3, 4]: continue
                    # TTA: Sliding window
                    offsets = [0, int(0.25*FS), int(0.5*FS)] if is_train else [0]
                    for off in offsets:
                        s = (start_idx - 1) + off
                        e = s + TIME_WINDOW
                        if e <= X_cnt.shape[0]:
                            raw_epoch = X_cnt[s:e, :22].T
                            filtered = butter_bandpass_filter(raw_epoch, 4.0, 38.0, FS, 4)
                            all_X.append(filtered)
                            all_y.append(label - 1)
            except: continue
        if len(all_X) == 0: return None, None
        return np.stack(all_X), np.array(all_y)
    except: return None, None

def get_dataloader(X, y, batch_size, shuffle=True):
    tensor_x = torch.Tensor(X).unsqueeze(1) # B, 1, 22, 1000
    tensor_y = torch.LongTensor(y)
    return DataLoader(TensorDataset(tensor_x, tensor_y), batch_size=batch_size, shuffle=shuffle)

def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for inputs, labels in loader:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0) # Safety clip
        optimizer.step()
        total_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    return total_loss / len(loader), 100 * correct / total

def evaluate_probs(model, loader):
    model.eval()
    all_probs, all_preds, all_labels = [], [], []
    with torch.no_grad():
        for inputs, labels in loader:
            inputs = inputs.to(DEVICE)
            outputs = model(inputs)
            probs = torch.softmax(outputs, dim=1)
            _, preds = torch.max(outputs, 1)
            all_probs.append(probs.cpu().numpy())
            all_preds.append(preds.cpu().numpy())
            all_labels.append(labels.numpy())
    return (np.vstack(all_probs), np.concatenate(all_preds), np.concatenate(all_labels))

# ==================== THE GOLD STANDARD MODEL ====================
#
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0).transpose(0, 1)
        self.register_buffer('pe', pe)

    def forward(self, x):
        return x + self.pe[:x.size(0), :]

class HybridCNNTransformer(nn.Module):
    def __init__(self, n_classes=4, n_channels=22, n_time=1000):
        super(HybridCNNTransformer, self).__init__()

        # 1. CNN Stem (The "Eyes"): Extracts features, reduces noise
        self.conv_stem = nn.Sequential(
            # Temporal Conv
            nn.Conv2d(1, 32, (1, 64), padding=(0, 32), bias=False),
            nn.BatchNorm2d(32),
            nn.ELU(),
            nn.AvgPool2d((1, 4)),
            nn.Dropout(0.5),

            # Spatial Conv
            nn.Conv2d(32, 64, (n_channels, 1), groups=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ELU(),
            nn.AvgPool2d((1, 4)),
            nn.Dropout(0.5)
        )

        # Calculate dims
        # Input: (1, 22, 1000) -> Conv1 -> (32, 22, 250) -> Conv2 -> (64, 1, 62)
        self.d_model = 64
        self.seq_len = 62

        # 2. Transformer Body (The "Brain"): Captures sequence context
        self.pos_encoder = PositionalEncoding(d_model=self.d_model)

        encoder_layers = nn.TransformerEncoderLayer(
            d_model=self.d_model,
            nhead=8,
            dim_feedforward=256,
            dropout=0.3,
            activation='gelu'
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layers, num_layers=4)

        # 3. Classification Head
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(self.d_model * self.seq_len, 256),
            nn.ELU(),
            nn.Dropout(0.5),
            nn.Linear(256, n_classes)
        )

    def forward(self, x):
        # x: (B, 1, 22, T)
        x = self.conv_stem(x)         # (B, 64, 1, 62)
        x = x.squeeze(2)              # (B, 64, 62)
        x = x.permute(2, 0, 1)        # (Seq, Batch, Dim) -> Required for PyTorch Transformer

        x = self.pos_encoder(x)
        x = self.transformer_encoder(x)

        x = x.permute(1, 0, 2)        # (Batch, Seq, Dim)
        x = self.head(x)
        return x

# ==================== EXECUTION LOOP ====================
print(f"\n{'='*60}")
print(f"STARTING {MODEL_NAME} - ROBUST & FAST")
print(f"{'='*60}")

# Data Cache
data_cache = {}
print("📥 Caching Raw Data...")
all_subjects = list(range(1, 10))
for s in all_subjects:
    X_tr, y_tr = load_bci_data_raw(s, DATA_PATH, True)
    X_te, y_te = load_bci_data_raw(s, DATA_PATH, False)
    if X_tr is not None:
        data_cache[s] = {'X_tr': X_tr, 'y_tr': y_tr, 'X_te': X_te, 'y_te': y_te}

results = {}
kappa_scores = {}

for target_sub in all_subjects:
    start_time = time.time()
    print(f"\n🎯 TARGET SUBJECT: {target_sub}")

    # Prepare Data
    X_source_list, y_source_list = [], []
    for src in all_subjects:
        if src != target_sub and src in data_cache:
            X_source_list.append(data_cache[src]['X_tr'])
            y_source_list.append(data_cache[src]['y_tr'])

    if not X_source_list: continue
    X_source = np.concatenate(X_source_list)
    y_source = np.concatenate(y_source_list)

    X_tgt_tr = data_cache[target_sub]['X_tr']
    y_tgt_tr = data_cache[target_sub]['y_tr']
    X_tgt_te = data_cache[target_sub]['X_te']
    y_tgt_te = data_cache[target_sub]['y_te']

    print("   ⚙️ Euclidean Alignment...")
    X_source = euclidean_alignment(X_source)
    X_tgt_tr = euclidean_alignment(X_tgt_tr)
    X_tgt_te = euclidean_alignment(X_tgt_te)

    scaler = StandardScaler()
    X_source = scale_data(X_source, scaler, fit=True)
    X_tgt_tr = scale_data(X_tgt_tr, scaler, fit=False)
    X_tgt_te = scale_data(X_tgt_te, scaler, fit=False)

    train_loader = get_dataloader(X_source, y_source, BATCH_SIZE)
    ft_loader = get_dataloader(X_tgt_tr, y_tgt_tr, 16) # Smaller batch for FT
    test_loader = get_dataloader(X_tgt_te, y_tgt_te, 16, shuffle=False)

    # --- TRAIN ---
    print(f"   🚀 Pre-training...")
    model = HybridCNNTransformer(n_classes=N_CLASSES).to(DEVICE)
    criterion = nn.CrossEntropyLoss() # Standard CE is sharper than smoothing here
    optimizer = optim.AdamW(model.parameters(), lr=LR_PRETRAIN, weight_decay=WEIGHT_DECAY)

    for ep in range(EPOCHS_PRETRAIN):
        loss, acc = train_epoch(model, train_loader, optimizer, criterion)
        if (ep+1) % 50 == 0:
            print(f"     Ep {ep+1}: Loss {loss:.4f} | Acc {acc:.2f}%")

    print(f"   🔧 Fine-tuning...")
    optimizer_ft = optim.AdamW(model.parameters(), lr=LR_FINETUNE, weight_decay=WEIGHT_DECAY)

    best_acc = 0.0
    best_state = copy.deepcopy(model.state_dict())

    for ep in range(EPOCHS_FINETUNE):
        loss, acc = train_epoch(model, ft_loader, optimizer_ft, criterion)
        if acc > best_acc:
            best_acc = acc
            best_state = copy.deepcopy(model.state_dict())
        if (ep+1) % 20 == 0:
            print(f"     FT Ep {ep+1}: Loss {loss:.4f} | Acc {acc:.2f}%")

    model.load_state_dict(best_state)

    # --- EVAL ---
    try:
        probs, preds, labels = evaluate_probs(model, test_loader)
        acc = 100 * (preds == labels).mean()
        kappa = cohen_kappa_score(labels, preds)

        results[target_sub] = acc
        kappa_scores[target_sub] = kappa
        print(f"   📊 S{target_sub}: Acc {acc:.2f}% | Kappa {kappa:.3f}")

        # Save
        s_path = os.path.join(SAVE_PATH, f"S{target_sub}")
        np.save(f"{s_path}_probs.npy", probs)
        np.save(f"{s_path}_labels.npy", labels)
        np.save(f"{s_path}_preds.npy", preds)
        np.save(f"{s_path}_metrics.npy", np.array([acc, kappa]))

    except Exception as e:
        print(f"   ❌ Eval Failed: {e}")

# ==================== RESULTS ====================
print(f"\n{'='*60}")
print(f"FINAL RESULTS: {MODEL_NAME}")
if results:
    accs = list(results.values())
    print(f"AVG: {np.mean(accs):.2f}% | STD: {np.std(accs):.2f}")
    with open(os.path.join(SAVE_PATH, 'summary.json'), 'w') as f:
        json.dump({'acc': results, 'kappa': kappa_scores, 'mean': np.mean(accs)}, f)
    print(f"✅ Saved to {SAVE_PATH}/")

✅ Device: cuda

STARTING Gold_Standard_Hybrid - ROBUST & FAST
📥 Caching Raw Data...

🎯 TARGET SUBJECT: 1
   ⚙️ Euclidean Alignment...
   ... Applying Euclidean Alignment (Shape: (6912, 22, 1000))
   ... Applying Euclidean Alignment (Shape: (864, 22, 1000))
   ... Applying Euclidean Alignment (Shape: (288, 22, 1000))
   🚀 Pre-training...


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


     Ep 50: Loss 0.3298 | Acc 87.77%
     Ep 100: Loss 0.1803 | Acc 94.02%
   🔧 Fine-tuning...
     FT Ep 20: Loss 0.5579 | Acc 78.47%
     FT Ep 40: Loss 0.3046 | Acc 88.66%
     FT Ep 60: Loss 0.2049 | Acc 92.82%
   📊 S1: Acc 75.69% | Kappa 0.676

🎯 TARGET SUBJECT: 2
   ⚙️ Euclidean Alignment...
   ... Applying Euclidean Alignment (Shape: (6912, 22, 1000))
   ... Applying Euclidean Alignment (Shape: (864, 22, 1000))
   ... Applying Euclidean Alignment (Shape: (288, 22, 1000))
   🚀 Pre-training...


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


     Ep 50: Loss 0.2886 | Acc 90.19%
     Ep 100: Loss 0.1743 | Acc 94.55%
   🔧 Fine-tuning...
     FT Ep 20: Loss 1.2230 | Acc 52.89%
     FT Ep 40: Loss 0.7933 | Acc 70.49%
     FT Ep 60: Loss 0.4746 | Acc 81.13%
   📊 S2: Acc 49.31% | Kappa 0.324

🎯 TARGET SUBJECT: 3
   ⚙️ Euclidean Alignment...
   ... Applying Euclidean Alignment (Shape: (6912, 22, 1000))
   ... Applying Euclidean Alignment (Shape: (864, 22, 1000))
   ... Applying Euclidean Alignment (Shape: (288, 22, 1000))
   🚀 Pre-training...


...

     Ep 50: Loss 0.3105 | Acc 89.13%
     Ep 100: Loss 0.1879 | Acc 93.75%
   🔧 Fine-tuning...
     FT Ep 20: Loss 0.9046 | Acc 66.55%
     FT Ep 40: Loss 0.4379 | Acc 83.91%
     FT Ep 60: Loss 0.3098 | Acc 87.85%
   📊 S7: Acc 72.57% | Kappa 0.634

🎯 TARGET SUBJECT: 8
   ⚙️ Euclidean Alignment...
   ... Applying Euclidean Alignment (Shape: (6912, 22, 1000))
   ... Applying Euclidean Alignment (Shape: (864, 22, 1000))
   ... Applying Euclidean Alignment (Shape: (288, 22, 1000))
   🚀 Pre-training...


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


     Ep 50: Loss 1.2825 | Acc 36.72%
     Ep 100: Loss 1.2307 | Acc 39.99%
   🔧 Fine-tuning...
     FT Ep 20: Loss 1.1958 | Acc 43.29%
     FT Ep 40: Loss 1.1650 | Acc 45.95%
     FT Ep 60: Loss 1.1650 | Acc 44.56%
   📊 S8: Acc 52.08% | Kappa 0.361

🎯 TARGET SUBJECT: 9
   ⚙️ Euclidean Alignment...
   ... Applying Euclidean Alignment (Shape: (6912, 22, 1000))
   ... Applying Euclidean Alignment (Shape: (864, 22, 1000))
   ... Applying Euclidean Alignment (Shape: (288, 22, 1000))
   🚀 Pre-training...


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


     Ep 50: Loss 0.2701 | Acc 90.51%
     Ep 100: Loss 0.1756 | Acc 94.08%
   🔧 Fine-tuning...
     FT Ep 20: Loss 0.4635 | Acc 85.07%
     FT Ep 40: Loss 0.2522 | Acc 91.44%
     FT Ep 60: Loss 0.1084 | Acc 96.30%
   📊 S9: Acc 76.04% | Kappa 0.681

FINAL RESULTS: Gold_Standard_Hybrid
AVG: 63.77% | STD: 11.57
✅ Saved to /content/gdrive/MyDrive/BCI_ENSEMBLE/Gold_Standard_Hybrid/
